# ARC Prize 2026 (ARC-AGI-2) — H1 "Pure Reasoning Ceiling"

**Branch** `h1/pure-xhigh` · **Kernel** `liangwanyiudavid/arc26-h1-pure-xhigh`

Hypothesis under test: **Qwen3.8-27B FP8 + `reasoning_effort=xhigh` + a minimal harness
reaches ≥ 50% on ARC-AGI-2**, with no judge model, no code-refinement loop, no test-time
training and no fine-tuning. The only levers are reasoning effort, sample count, CSV grid
encoding, tolerant parsing and shape-diverse two-attempt selection.

Full quantitative budget model, score prediction and falsification criteria: `HYPOTHESIS.md`.

### What this kernel does

1. Writes the `harness/` package to disk (stdlib-only, 223 unit tests green offline).
2. Installs vLLM — from a pre-baked wheelhouse if attached, else `pip` (needs internet, i.e. a non-rerun save).
3. Launches one vLLM instance, `--tensor-parallel-size 4`, through a **6-rung flag fallback ladder** so a version-sensitive flag cannot kill the run.
4. Probes `reasoning_effort=xhigh` and **calibrates** the per-sample token cost from real output.
5. **Phase A:** 2 × `low`-effort samples on *all* tasks (~3.7% of budget, ~23 min) → a complete, schema-valid `submission.json` exists from that point on.
6. **Phase B:** xhigh **waves** over all tasks (never task-by-task — serial execution would overrun 12 h by ~1.9×), with consensus early-stop and a governor that re-plans every wave against *measured* throughput.
7. Cumulative atomic submission write after every wave + an independent watchdog thread.

### Attachments required

| Type | Slug | Why |
|---|---|---|
| Competition | `arc-prize-2026-arc-agi-2` | provides `arc-agi_test_challenges.json` at rerun |
| Dataset | `saltb0x/qwen3-8-27b-fp8` | 25.3 GB official FP8 HF checkpoint (verified layout) |
| Dataset *(optional)* | `jakobbrggen/qwen3-8-27b-fp8-hf-snapshot` | interchangeable FP8 mirror; auto-detected as a fallback |
| Dataset *(optional)* | a pre-baked vLLM wheelhouse | **required for the offline competition rerun** |

> **Do NOT use `sergiodefreitas/qwen3-8-27b`** — it is an Ollama GGUF split archive
> (`ollama_weights.part_aa`, …), not an HF checkpoint. vLLM cannot load it.
> **Do NOT use an NVFP4 mirror** — NVFP4 is Blackwell-native; L4 is Ada (sm_89).

### Metadata

`enable_gpu` is **`false`** in `kernel-metadata.json`: this kernel was authored and pushed
without executing it, because running it consumes paid L4x4 quota. **Enable the L4x4
accelerator in the Kaggle UI before running.** In non-rerun mode it scores itself against
the 120-task evaluation split; in competition rerun it uses the 240 unseen test tasks.


## 1. Write the harness to disk

Every module below is the exact source that the offline test-suite (`pytest tests`, 223 tests) exercises. The harness itself is **standard-library only** -- the sole third-party dependency in this kernel is vLLM.

The first cell creates `harness/` because IPython's `%%writefile` magic does **not** create parent directories and raises `FileNotFoundError` if they are missing.


In [ ]:
import os
os.makedirs('harness', exist_ok=True)
print('harness/ ready:', os.path.isdir('harness'))


In [ ]:
%%writefile harness/__init__.py
"""H1 "Pure Reasoning Ceiling" harness for ARC Prize 2026 (ARC-AGI-2).

Minimal, clean, no judge model, no code refinement, no test-time training. The only
levers are `reasoning_effort`, sample count, CSV grid encoding, tolerant parsing and
shape-diverse two-attempt selection.
"""

from __future__ import annotations

from .budget import BudgetConfig, BudgetGovernor, Clock, ManualClock, SystemClock, TokenLedger
from .config import HarnessConfig, SamplingConfig, ServingConfig
from .diagnostics import CoverageStats, CoverageTracker
from .grids import (
    Grid,
    fallback_grid,
    from_key,
    is_grid,
    parse_csv_strict,
    parse_grid_blocks,
    parse_last_grid,
    shape,
    to_csv,
    to_csv_indented,
    to_key,
)
from .llm import (
    EFFORT_RANK,
    VALID_EFFORTS,
    ChatRequest,
    ChatResponse,
    EffortError,
    HTTPChatClient,
    LLMClient,
    MockLLM,
    RoundRobinClient,
    Usage,
    validate_effort,
)
from .parse import AnswerParse, merge_reasoning_and_content, parse_answers
from .prompts import Prompt, build_messages, build_prompt, prompt_token_estimate
from .runner import RunOutcome, budget_table, load_tasks, mock_answers, run_harness
from .scoring import ScoreReport, TaskScore, score_predictions, score_submission
from .select import Candidate, CandidatePool, IndexResult, consensus_reached, task_consensus
from .submission import SubmissionWriter, build_entry, empty_submission, validate_submission_obj
from .tasks import ARCTask, load_challenges, resolve_data_dir, total_test_outputs
from .watchdog import Watchdog
from .waves import RunReport, Scheduler, TaskState, WaveRecord

__all__ = [
    "AnswerParse",
    "ARCTask",
    "BudgetConfig",
    "BudgetGovernor",
    "Candidate",
    "CandidatePool",
    "ChatRequest",
    "ChatResponse",
    "Clock",
    "CoverageStats",
    "CoverageTracker",
    "EffortError",
    "EFFORT_RANK",
    "Grid",
    "HTTPChatClient",
    "HarnessConfig",
    "IndexResult",
    "LLMClient",
    "ManualClock",
    "MockLLM",
    "RoundRobinClient",
    "Prompt",
    "RunOutcome",
    "RunReport",
    "SamplingConfig",
    "ScoreReport",
    "Scheduler",
    "ServingConfig",
    "SubmissionWriter",
    "SystemClock",
    "TaskScore",
    "TaskState",
    "TokenLedger",
    "Usage",
    "VALID_EFFORTS",
    "Watchdog",
    "WaveRecord",
    "budget_table",
    "build_entry",
    "build_messages",
    "build_prompt",
    "consensus_reached",
    "empty_submission",
    "fallback_grid",
    "from_key",
    "is_grid",
    "load_challenges",
    "load_tasks",
    "merge_reasoning_and_content",
    "mock_answers",
    "parse_answers",
    "parse_csv_strict",
    "parse_grid_blocks",
    "parse_last_grid",
    "prompt_token_estimate",
    "resolve_data_dir",
    "run_harness",
    "score_predictions",
    "score_submission",
    "shape",
    "task_consensus",
    "to_csv",
    "to_csv_indented",
    "to_key",
    "total_test_outputs",
    "validate_effort",
    "validate_submission_obj",
]


In [ ]:
%%writefile harness/grids.py
"""Grid representation, CSV rendering, and tolerant parsing.

CSV is the grid encoding Land benchmarked best out of 9 (`0,1,2\\n3,4,5`); a bad
encoding costs roughly 10% of score. Everything here is pure stdlib so it runs
identically on a Windows dev box and inside a Kaggle rerun container.
"""

from __future__ import annotations

import re
from collections.abc import Sequence as _Seq
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

Grid = List[List[int]]
GridKey = Tuple[Tuple[int, ...], ...]

MIN_CELL = 0
MAX_CELL = 9
MAX_PROSE_GAP = 1


def is_grid(obj: object) -> bool:
    if isinstance(obj, (str, bytes)) or not isinstance(obj, _Seq):
        return False
    if len(obj) == 0:
        return False
    width: Optional[int] = None
    for row in obj:
        if isinstance(row, (str, bytes)) or not isinstance(row, _Seq) or len(row) == 0:
            return False
        if width is None:
            width = len(row)
        elif len(row) != width:
            return False
        for cell in row:
            if isinstance(cell, bool) or not isinstance(cell, int):
                return False
            if cell < MIN_CELL or cell > MAX_CELL:
                return False
    return True


def to_key(grid: Grid) -> GridKey:
    return tuple(tuple(int(c) for c in row) for row in grid)


def from_key(key: GridKey) -> Grid:
    return [list(row) for row in key]


def shape(grid: Grid) -> Tuple[int, int]:
    return (len(grid), len(grid[0]) if grid else 0)


def to_csv(grid: Grid) -> str:
    return "\n".join(",".join(str(int(c)) for c in row) for row in grid)


def to_csv_indented(grid: Grid, padding: str = "    ") -> str:
    return "\n".join(padding + ",".join(str(int(c)) for c in row) for row in grid)


def parse_csv_strict(text: str) -> Optional[Grid]:
    lines = [ln for ln in text.strip().splitlines() if ln.strip() != ""]
    if not lines:
        return None
    grid: Grid = []
    width: Optional[int] = None
    for line in lines:
        tokens = line.strip().split(",")
        if not tokens or not all(t.strip().isdigit() for t in tokens):
            return None
        row = [int(t.strip()) for t in tokens]
        if width is None:
            width = len(row)
        elif len(row) != width:
            return None
        if any(c < MIN_CELL or c > MAX_CELL for c in row):
            return None
        grid.append(row)
    return grid or None


_CSV_ROW_RE = re.compile(r"^[\s`]*((?:\d{1,2}\s*,)+\s*\d{1,2})[\s`]*$")
_LEADING_ENUM_RE = re.compile(r"^\s*(?:\d+\s*[\.\)]|[-*+]|\**Row\s*\d+\**?\s*:?)\s*", re.IGNORECASE)


def _candidate_row(line: str) -> Optional[List[int]]:
    raw = line.strip().strip("`")
    if not raw:
        return None
    m = _CSV_ROW_RE.match(raw)
    if not m:
        return None
    body = m.group(1).replace("`", "")
    tokens = [t.strip() for t in body.split(",")]
    if not tokens or not all(t.isdigit() for t in tokens):
        return None
    row = [int(t) for t in tokens]
    if any(c < MIN_CELL or c > MAX_CELL for c in row):
        return None
    return row


def _row_after_label(line: str) -> Optional[List[int]]:
    stripped = _LEADING_ENUM_RE.sub("", line.strip())
    stripped = re.sub(r"^\**\s*(?:Output|Answer|Result|Grid)\s*\d*\**\s*:\s*", "", stripped, flags=re.IGNORECASE)
    if ":" in stripped:
        stripped = stripped.split(":", 1)[1].strip()
    stripped = stripped.strip().strip("`").strip()
    stripped = stripped.lstrip("[").rstrip("]").strip()
    if not stripped:
        return None
    tokens = [t.strip() for t in stripped.split(",")]
    if len(tokens) < 2 or not all(t.isdigit() for t in tokens):
        return None
    row = [int(t) for t in tokens]
    if any(c < MIN_CELL or c > MAX_CELL for c in row):
        return None
    return row


def _mode_int(values: Iterable[int]) -> int:
    counts: Dict[int, int] = {}
    for v in values:
        counts[v] = counts.get(v, 0) + 1
    return max(sorted(counts.items()), key=lambda kv: kv[1])[0]


def _mode_shape(values: Sequence[Tuple[int, int]]) -> Tuple[int, int]:
    counts: Dict[Tuple[int, int], int] = {}
    for v in values:
        counts[v] = counts.get(v, 0) + 1
    return max(sorted(counts.items()), key=lambda kv: kv[1])[0]


def regularize(rows: Sequence[List[int]]) -> Optional[Grid]:
    """Repair a mildly ragged row set into a rectangular grid.

    Models occasionally drop a trailing comma or let a prose line leak in. We keep
    the modal width and discard rows that do not match it, accepting only if the
    surviving rows are a strict majority. Short rows are never padded: inventing
    cells would fabricate an answer.
    """
    if not rows:
        return None
    widths = [len(r) for r in rows]
    width = _mode_int(widths)
    kept = [list(r) for r in rows if len(r) == width]
    if not kept:
        return None
    if len(kept) * 2 <= len(rows):
        return None
    return kept


_SINGLE_CELL_RE = re.compile(r"^\d$")


def parse_grid_blocks(text: str, allow_single_cell: bool = False) -> List[Grid]:
    """Return every plausible CSV grid block in `text`, in order of appearance.

    Blocks break on markdown fences, blank lines, and large width jumps; a single
    interleaved prose line is tolerated. `allow_single_cell` additionally accepts a
    lone digit as a 1x1 grid, which is only safe in a slice already delimited by an
    explicit answer marker.
    """
    if not text:
        return []
    lines = text.splitlines()
    rows_at: List[Tuple[int, List[int]]] = []
    fence_lines: List[int] = []

    for idx, line in enumerate(lines):
        if line.strip().startswith("```"):
            fence_lines.append(idx)
            continue
        row = _candidate_row(line)
        if row is None:
            row = _row_after_label(line)
        if row is None and allow_single_cell:
            m = _SINGLE_CELL_RE.match(line.strip())
            if m:
                row = [int(m.group(0))]
        if row is not None:
            rows_at.append((idx, row))

    if not rows_at:
        return []

    def split_between(a: int, b: int) -> bool:
        """True when lines a and b belong to different grid blocks."""
        gap = b - a - 1
        if any(a < f < b for f in fence_lines):
            return True
        if gap <= 0:
            return False
        intervening = lines[a + 1 : b]
        if all(ln.strip() == "" for ln in intervening):
            return True
        return gap > MAX_PROSE_GAP

    blocks: List[List[Tuple[int, List[int]]]] = []
    current: List[Tuple[int, List[int]]] = [rows_at[0]]
    for prev, cur in zip(rows_at, rows_at[1:]):
        width_jump = abs(len(cur[1]) - len(current[0][1])) > 2
        if width_jump or split_between(prev[0], cur[0]):
            blocks.append(current)
            current = [cur]
        else:
            current.append(cur)
    blocks.append(current)

    out: List[Grid] = []
    for block in blocks:
        grid = regularize([row for _, row in block])
        if grid is not None:
            out.append(grid)
    return out


def parse_last_grid(text: str) -> Optional[Grid]:
    blocks = parse_grid_blocks(text)
    return blocks[-1] if blocks else None


def fallback_grid(test_input: Grid, train: Sequence[Tuple[Grid, Grid]]) -> Grid:
    """Deterministic last-resort answer for when nothing parsed.

    Uses the modal training-output shape filled with the modal non-background
    training-output colour; degrades to the identity transform. Any deterministic
    guess beats an empty grid, which can never match.
    """
    if train:
        target = _mode_shape([shape(o) for _, o in train])
        colors = [c for _, o in train for row in o for c in row if c != 0]
        fill = _mode_int(colors) if colors else 0
        if 1 <= target[0] <= 30 and 1 <= target[1] <= 30:
            return [[fill] * target[1] for _ in range(target[0])]
    return [list(row) for row in test_input]


def grids_equal(a: Grid, b: Grid) -> bool:
    return to_key(a) == to_key(b)


In [ ]:
%%writefile harness/tasks.py
"""ARC task loading, container types, and Kaggle path discovery."""

from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

from .grids import Grid, is_grid

DEV_DATA_DIR = r"C:\Users\daive\Downloads\arc-agi-2\data"
KAGGLE_COMP_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
KAGGLE_WORKING = "/kaggle/working"

SPLITS = ("training", "evaluation", "test")


@dataclass(frozen=True)
class ARCTask:
    task_id: str
    train: Tuple[Tuple[Grid, Grid], ...]
    test: Tuple[Grid, ...]
    solutions: Optional[Tuple[Grid, ...]] = field(default=None, compare=False)

    @property
    def n_test(self) -> int:
        return len(self.test)

    def prompt_chars(self) -> int:
        total = 0
        for a, b in self.train:
            total += _csv_chars(a) + _csv_chars(b)
        for g in self.test:
            total += _csv_chars(g)
        return total

    def approx_prompt_tokens(self, chars_per_token: float = 3.2, overhead: int = 320) -> int:
        return int(self.prompt_chars() / chars_per_token) + overhead


def _csv_chars(grid: Grid) -> int:
    return sum(len(",".join(map(str, row))) for row in grid) + max(0, len(grid) - 1)


def _validate_raw(task_id: str, raw: dict) -> Tuple[Tuple[Tuple[Grid, Grid], ...], Tuple[Grid, ...]]:
    train_raw = raw.get("train") or []
    test_raw = raw.get("test") or []
    if not isinstance(train_raw, list) or not isinstance(test_raw, list):
        raise ValueError(f"{task_id}: train/test must be lists")
    train: List[Tuple[Grid, Grid]] = []
    for i, ex in enumerate(train_raw):
        gi, go = ex.get("input"), ex.get("output")
        if not is_grid(gi) or not is_grid(go):
            raise ValueError(f"{task_id}: bad train pair {i}")
        train.append((gi, go))
    test: List[Grid] = []
    for i, ex in enumerate(test_raw):
        gi = ex.get("input")
        if not is_grid(gi):
            raise ValueError(f"{task_id}: bad test input {i}")
        test.append(gi)
    if not test:
        raise ValueError(f"{task_id}: no test inputs")
    return tuple(train), tuple(test)


def load_challenges(path: str, solutions_path: Optional[str] = None) -> List[ARCTask]:
    with open(path, "r", encoding="utf-8") as fh:
        blob: Dict[str, dict] = json.load(fh)
    sols: Dict[str, List[Grid]] = {}
    if solutions_path and os.path.exists(solutions_path):
        with open(solutions_path, "r", encoding="utf-8") as fh:
            sols = json.load(fh)

    tasks: List[ARCTask] = []
    for task_id in sorted(blob.keys()):
        train, test = _validate_raw(task_id, blob[task_id])
        gt: Optional[Tuple[Grid, ...]] = None
        if task_id in sols:
            raw_sols = sols[task_id]
            if isinstance(raw_sols, list) and all(is_grid(g) for g in raw_sols) and len(raw_sols) == len(test):
                gt = tuple(raw_sols)
        tasks.append(ARCTask(task_id=task_id, train=train, test=test, solutions=gt))
    return tasks


def task_ids(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as fh:
        return sorted(json.load(fh).keys())


def resolve_data_dir(explicit: Optional[str] = None) -> str:
    for cand in (
        explicit,
        os.environ.get("ARC_DATA_DIR"),
        KAGGLE_COMP_DIR,
        os.path.join(os.getcwd(), "data"),
        DEV_DATA_DIR,
    ):
        if cand and os.path.isdir(cand):
            return cand
    raise FileNotFoundError("could not locate ARC data directory")


def resolve_challenges_path(data_dir: str, split: str = "test") -> str:
    if split not in SPLITS:
        raise ValueError(f"unknown split {split!r}")
    return os.path.join(data_dir, f"arc-agi_{split}_challenges.json")


def resolve_solutions_path(data_dir: str, split: str) -> Optional[str]:
    path = os.path.join(data_dir, f"arc-agi_{split}_solutions.json")
    return path if os.path.exists(path) else None


def resolve_output_dir(explicit: Optional[str] = None) -> str:
    for cand in (explicit, os.environ.get("ARC_OUTPUT_DIR")):
        if cand:
            return cand
    if os.path.isdir(KAGGLE_WORKING):
        return KAGGLE_WORKING
    return os.path.join(os.getcwd(), "output")


def is_kaggle_rerun() -> bool:
    return bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))


def total_test_outputs(tasks: Sequence[ARCTask]) -> int:
    return sum(t.n_test for t in tasks)


In [ ]:
%%writefile harness/prompts.py
"""Prompt construction: minimal, clean, CSV-encoded.

H1 deliberately avoids prescriptive multi-section templates. Land's documented
negative result is that "prescriptive prompting templates and iterative
refinement systematically reduce hypothesis diversity and degrade performance",
so the prompt states the task contract and the output format and then gets out of
the model's way.
"""

from __future__ import annotations

import random
from dataclasses import dataclass
from typing import List, Optional, Sequence

from .grids import Grid, to_csv_indented
from .tasks import ARCTask

SYSTEM_PROMPT = """You are an expert at ARC-AGI-2 abstract reasoning puzzles.

You are shown input/output grid pairs from one puzzle. Grids are rectangular
matrices of integers 0-9 written as comma-separated rows; 0 is usually but not
always the background. Exactly one deterministic transformation maps every input
grid to its output grid. Infer that transformation from the training pairs, check
that it reproduces every one of them exactly, then apply it to the test inputs.

Be careful about these things, which decide most puzzles:
- The output size is set by the rule, not by the input size. Determine it explicitly.
- Distinguish the relevant objects from the background before transforming anything.
- A rule that explains all training pairs exactly is far more likely to be right
  than one that explains most of them.

Answer format. After your reasoning, write the answer for each test input like this:

ANSWER 1:
2,0,1
0,2,0
1,0,2

Use one comma-separated row per line with no spaces and no brackets. If there is
more than one test input, continue with ANSWER 2:, ANSWER 3:, and so on, in the
same order as the test inputs were given. Write nothing after the final answer."""

TRAIN_HEADER = "Training pair {i} input:\n{gi}\n\nTraining pair {i} output:\n{go}"
TEST_HEADER = "Test input {i}:\n{gi}"
FOOTER = (
    "Work out the transformation and give the output grid for each of the {n} test "
    "input{s}. Reply with ANSWER 1:{extra} in the exact format shown above."
)


@dataclass(frozen=True)
class Prompt:
    system: str
    user: str

    @property
    def approx_tokens(self) -> int:
        return int((len(self.system) + len(self.user)) / 3.2)


def build_prompt(
    task: ARCTask,
    shuffle_train: bool = False,
    seed: int = 0,
    max_train_pairs: Optional[int] = None,
) -> Prompt:
    pairs: List[tuple] = list(task.train)
    if max_train_pairs is not None and len(pairs) > max_train_pairs:
        pairs = pairs[:max_train_pairs]
    if shuffle_train and len(pairs) > 1:
        rng = random.Random(f"{seed}:{task.task_id}")
        rng.shuffle(pairs)

    parts: List[str] = []
    for i, (gi, go) in enumerate(pairs, start=1):
        parts.append(
            TRAIN_HEADER.format(
                i=i, gi=to_csv_indented(gi, "    "), go=to_csv_indented(go, "    ")
            )
        )
    for i, gi in enumerate(task.test, start=1):
        parts.append(TEST_HEADER.format(i=i, gi=to_csv_indented(gi, "    ")))

    n = task.n_test
    extra = "".join(f" ANSWER {k}:" for k in range(2, n + 1))
    parts.append(FOOTER.format(n=n, s="" if n == 1 else "s", extra=extra))
    return Prompt(system=SYSTEM_PROMPT, user="\n\n".join(parts))


def build_messages(task: ARCTask, shuffle_train: bool = False, seed: int = 0) -> List[dict]:
    prompt = build_prompt(task, shuffle_train=shuffle_train, seed=seed)
    return [
        {"role": "system", "content": prompt.system},
        {"role": "user", "content": prompt.user},
    ]


def prompt_token_estimate(tasks: Sequence[ARCTask]) -> dict:
    vals = sorted(t.approx_prompt_tokens() for t in tasks)
    if not vals:
        return {"n": 0, "mean": 0, "p50": 0, "p90": 0, "max": 0, "total": 0}
    return {
        "n": len(vals),
        "mean": int(sum(vals) / len(vals)),
        "p50": vals[len(vals) // 2],
        "p90": vals[min(len(vals) - 1, int(0.9 * len(vals)))],
        "max": vals[-1],
        "total": sum(vals),
    }


In [ ]:
%%writefile harness/parse.py
"""Tolerant extraction of ANSWER blocks from free-form thinking-model output.

Structured output (`guided_json` / `response_format`) is unreliable with vLLM
thinking mode (vLLM#35700) and the reasoning parser splits text into
`reasoning_content` and `content`, so we generate free-form and parse tolerantly.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

from .grids import Grid, parse_grid_blocks

_MARKER_RE = re.compile(
    r"(?:^|\n)[^\n]{0,40}?\b(?:ANSWER|OUTPUT|RESULT|SOLUTION)\s*#?\s*(\d{1,2})\s*\**\s*[:\-]",
    re.IGNORECASE,
)


@dataclass
class AnswerParse:
    grids: Dict[int, Grid] = field(default_factory=dict)
    method: str = "none"
    n_blocks: int = 0

    @property
    def ok(self) -> bool:
        return bool(self.grids)

    def indices(self) -> List[int]:
        return sorted(self.grids.keys())

    def as_list(self, n_test: int) -> List[Optional[Grid]]:
        return [self.grids.get(i + 1) for i in range(n_test)]


def _marker_positions(text: str) -> List[Tuple[int, int, int]]:
    """Return (index_label, slice_start, slice_end) for every answer marker."""
    hits = [(m.group(0), int(m.group(1)), m.end()) for m in _MARKER_RE.finditer(text)]
    out: List[Tuple[int, int, int]] = []
    for pos, (_full, label, end) in enumerate(hits):
        nxt = hits[pos + 1][2] if pos + 1 < len(hits) else len(text)
        out.append((label, end, max(end, nxt)))
    return out


def parse_answers(text: str, n_test: int) -> AnswerParse:
    if not text:
        return AnswerParse(method="empty")

    markers = _marker_positions(text)
    if markers:
        by_label: Dict[int, List[Tuple[int, int]]] = {}
        for label, start, end in markers:
            by_label.setdefault(label, []).append((start, end))

        grids: Dict[int, Grid] = {}
        for label in sorted(by_label):
            if label < 1 or label > n_test:
                continue
            for start, end in reversed(by_label[label]):
                blocks = parse_grid_blocks(text[start:end], allow_single_cell=True)
                if blocks:
                    grids[label] = blocks[0]
                    break
        if grids:
            return AnswerParse(
                grids=grids, method="marker", n_blocks=len(parse_grid_blocks(text))
            )

    blocks = parse_grid_blocks(text)
    if not blocks:
        return AnswerParse(method="none")

    if n_test == 1:
        return AnswerParse(grids={1: blocks[-1]}, method="last-block", n_blocks=len(blocks))

    if len(blocks) == n_test:
        return AnswerParse(
            grids={i + 1: g for i, g in enumerate(blocks)},
            method="ordered-blocks",
            n_blocks=len(blocks),
        )

    if len(blocks) >= n_test:
        tail = blocks[-n_test:]
        return AnswerParse(
            grids={i + 1: g for i, g in enumerate(tail)},
            method="tail-blocks",
            n_blocks=len(blocks),
        )

    return AnswerParse(grids={1: blocks[-1]}, method="last-block", n_blocks=len(blocks))


def merge_reasoning_and_content(reasoning: str, content: str) -> str:
    """Build the search space for the parser.

    With `--reasoning-parser qwen3` the visible answer lands in `content`, but a
    truncated generation (`finish_reason == "length"`) can leave the answer inside
    the reasoning stream, so both are searched, content first.
    """
    parts = [p for p in (content or "", reasoning or "") if p]
    return "\n\n".join(parts)


In [ ]:
%%writefile harness/llm.py
"""LLM client interface: OpenAI-compatible HTTP (vLLM) plus an offline MockLLM.

Every component downstream of `LLMClient` is backend-agnostic, which is what makes
the whole harness testable with no GPU. The HTTP client uses only `urllib` from the
standard library so the Kaggle wheelhouse needs nothing beyond vLLM itself.
"""

from __future__ import annotations

import itertools
import json
import random
import time
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

from .grids import Grid, to_csv

VALID_EFFORTS: Tuple[str, ...] = ("xhigh", "medium", "low")
EFFORT_RANK = {"low": 0, "medium": 1, "xhigh": 2}


class EffortError(ValueError):
    """Raised for a reasoning_effort value the Qwen3.8 chat template rejects."""


def validate_effort(effort: str) -> str:
    if effort not in VALID_EFFORTS:
        raise EffortError(
            f"reasoning_effort={effort!r} is invalid; the Qwen3.8 chat template "
            f"accepts only {VALID_EFFORTS} and raises HTTP 500 otherwise "
            f"(QwenLM/Qwen3.8#217)"
        )
    return effort


@dataclass
class ChatRequest:
    messages: List[Dict[str, str]]
    max_tokens: int = 16384
    temperature: float = 0.9
    top_p: float = 0.95
    seed: Optional[int] = None
    reasoning_effort: str = "xhigh"
    meta: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self) -> None:
        validate_effort(self.reasoning_effort)


@dataclass
class Usage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    reasoning_tokens: int = 0

    @property
    def total(self) -> int:
        return self.prompt_tokens + self.completion_tokens

    def as_dict(self) -> Dict[str, int]:
        return {
            "prompt_tokens": self.prompt_tokens,
            "completion_tokens": self.completion_tokens,
            "reasoning_tokens": self.reasoning_tokens,
        }


@dataclass
class ChatResponse:
    content: str = ""
    reasoning: str = ""
    usage: Usage = field(default_factory=Usage)
    finish_reason: str = ""
    elapsed: float = 0.0
    error: Optional[str] = None
    attempts: int = 1
    meta: Dict[str, Any] = field(default_factory=dict)

    @property
    def ok(self) -> bool:
        return self.error is None

    @property
    def truncated(self) -> bool:
        return self.finish_reason == "length"


class LLMClient:
    """Interface every backend implements."""

    name = "base"

    def complete(self, request: ChatRequest) -> ChatResponse:
        raise NotImplementedError

    def health(self) -> bool:
        return True

    def close(self) -> None:
        return None


class HTTPChatClient(LLMClient):
    """Blocking OpenAI-compatible `/v1/chat/completions` client.

    Concurrency is supplied by the caller's thread pool, which keeps the runtime
    dependency list empty and makes each request independently cancellable by the
    budget governor.
    """

    name = "vllm-http"

    def __init__(
        self,
        base_url: str = "http://127.0.0.1:8000/v1",
        model: str = "qwen3.8-27b",
        api_key: str = "EMPTY",
        timeout: float = 1800.0,
        max_retries: int = 3,
        backoff: float = 2.0,
        effort_mechanism: str = "chat_template_kwargs",
        extra_payload: Optional[Dict[str, Any]] = None,
        logger: Optional[Callable[[str], None]] = None,
    ) -> None:
        if effort_mechanism not in ("chat_template_kwargs", "top_level", "none"):
            raise ValueError(f"unknown effort_mechanism {effort_mechanism!r}")
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.api_key = api_key
        self.timeout = timeout
        self.max_retries = max_retries
        self.backoff = backoff
        self.effort_mechanism = effort_mechanism
        self.extra_payload = dict(extra_payload or {})
        self._log = logger or (lambda _m: None)

    def _payload(self, request: ChatRequest) -> Dict[str, Any]:
        payload: Dict[str, Any] = {
            "model": self.model,
            "messages": request.messages,
            "max_tokens": request.max_tokens,
            "temperature": request.temperature,
            "top_p": request.top_p,
            "stream": False,
        }
        if request.seed is not None:
            payload["seed"] = request.seed
        if self.effort_mechanism == "chat_template_kwargs":
            payload["chat_template_kwargs"] = {"reasoning_effort": request.reasoning_effort}
        elif self.effort_mechanism == "top_level":
            payload["reasoning_effort"] = request.reasoning_effort
        payload.update(self.extra_payload)
        return payload

    def complete(self, request: ChatRequest) -> ChatResponse:
        payload = self._payload(request)
        body = json.dumps(payload).encode("utf-8")
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }
        url = f"{self.base_url}/chat/completions"
        started = time.monotonic()
        last_error = "unknown"

        for attempt in range(1, self.max_retries + 1):
            try:
                req = urllib.request.Request(url, data=body, headers=headers, method="POST")
                with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                    raw = json.loads(resp.read().decode("utf-8"))
                return self._parse(raw, time.monotonic() - started, attempt, request.meta)
            except urllib.error.HTTPError as exc:
                detail = ""
                try:
                    detail = exc.read().decode("utf-8", "replace")[:400]
                except Exception:
                    pass
                last_error = f"HTTP {exc.code}: {detail}"
                if exc.code in (400, 404, 422):
                    break
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
            if attempt < self.max_retries:
                time.sleep(self.backoff ** attempt)

        return ChatResponse(
            error=last_error,
            elapsed=time.monotonic() - started,
            attempts=self.max_retries,
            meta=dict(request.meta),
        )

    def _parse(
        self, raw: Dict[str, Any], elapsed: float, attempts: int, meta: Dict[str, Any]
    ) -> ChatResponse:
        choices = raw.get("choices") or [{}]
        message = choices[0].get("message") or {}
        usage_raw = raw.get("usage") or {}
        details = usage_raw.get("completion_tokens_details") or {}
        content = message.get("content")
        if content is None:
            content = ""
        elif isinstance(content, list):
            content = "".join(
                part.get("text", "") for part in content if isinstance(part, dict)
            )
        reasoning = message.get("reasoning_content") or message.get("reasoning") or ""
        usage = Usage(
            prompt_tokens=int(usage_raw.get("prompt_tokens") or 0),
            completion_tokens=int(usage_raw.get("completion_tokens") or 0),
            reasoning_tokens=int(
                details.get("reasoning_tokens") or usage_raw.get("reasoning_tokens") or 0
            ),
        )
        return ChatResponse(
            content=str(content),
            reasoning=str(reasoning),
            usage=usage,
            finish_reason=str(choices[0].get("finish_reason") or ""),
            elapsed=elapsed,
            attempts=attempts,
            meta=dict(meta),
        )

    def health(self) -> bool:
        url = f"{self.base_url}/models"
        try:
            req = urllib.request.Request(
                url, headers={"Authorization": f"Bearer {self.api_key}"}
            )
            with urllib.request.urlopen(req, timeout=10) as resp:
                return resp.status == 200
        except Exception:
            return False

    def wait_healthy(self, deadline_ts: float, poll: float = 5.0) -> bool:
        while time.monotonic() < deadline_ts:
            if self.health():
                return True
            time.sleep(poll)
        return self.health()


class RoundRobinClient(LLMClient):
    """Fans requests across several endpoints.

    Used for a 2 x (TP=2) serving topology: FP8 27B needs 13.5 GB/GPU at TP=2, which
    leaves ~6.6 GB for KV, so two independent 2-GPU instances each run a shallower
    batch but the pair recovers most of the aggregate throughput of a single TP=4
    instance while halving the blast radius of an NCCL failure.
    """

    name = "round-robin"

    def __init__(self, clients: Sequence[LLMClient]) -> None:
        if not clients:
            raise ValueError("RoundRobinClient needs at least one client")
        self.clients: List[LLMClient] = list(clients)
        self._counter = itertools.count()

    def complete(self, request: ChatRequest) -> ChatResponse:
        index = next(self._counter) % len(self.clients)
        return self.clients[index].complete(request)

    def health(self) -> bool:
        return any(c.health() for c in self.clients)

    def healthy(self) -> List[LLMClient]:
        return [c for c in self.clients if c.health()]

    def close(self) -> None:
        for client in self.clients:
            client.close()


class MockLLM(LLMClient):
    """Deterministic offline backend used by the test-suite and `eval_local --mock`.

    `answers` maps task_id -> list of ground-truth output grids. Each call returns
    the correct grids with probability `correct_prob`, else a deterministic
    perturbation, so voting, shape-diverse selection and budget accounting all get
    exercised on realistic-ish inputs.
    """

    name = "mock"

    def __init__(
        self,
        answers: Optional[Dict[str, Sequence[Grid]]] = None,
        correct_prob: float = 0.5,
        malformed_prob: float = 0.0,
        thinking_tokens: Optional[Dict[str, int]] = None,
        latency: float = 0.0,
        seed: int = 1234,
        tok_per_sec: float = 400.0,
        answer_fn: Optional[Callable[[ChatRequest], Optional[List[Grid]]]] = None,
    ) -> None:
        self.answers: Dict[str, List[Grid]] = {k: [list(r) for r in v] for k, v in (answers or {}).items()}
        self.correct_prob = correct_prob
        self.malformed_prob = malformed_prob
        self.thinking_tokens = dict(thinking_tokens or {"low": 900, "medium": 4000, "xhigh": 10000})
        self.latency = latency
        self.tok_per_sec = tok_per_sec
        self.rng = random.Random(seed)
        self.answer_fn = answer_fn
        self.calls: List[ChatRequest] = []
        self.tokens_emitted = 0

    def _perturb(self, grid: Grid, salt: int) -> Grid:
        rng = random.Random(salt)
        out = [list(row) for row in grid]
        for _ in range(max(1, len(out) * len(out[0]) // 8)):
            r = rng.randrange(len(out))
            c = rng.randrange(len(out[0]))
            out[r][c] = rng.randrange(10)
        return out

    def _render(self, grids: List[Grid]) -> str:
        lines = ["<think>", "Let me work through the transformation step by step.", "</think>"]
        for i, g in enumerate(grids, start=1):
            lines.append("")
            lines.append(f"ANSWER {i}:")
            lines.append(to_csv(g))
        return "\n".join(lines)

    def complete(self, request: ChatRequest) -> ChatResponse:
        started = time.monotonic()
        self.calls.append(request)
        task_id = str(request.meta.get("task_id", ""))
        truth = self.answers.get(task_id)

        if self.answer_fn is not None:
            grids = self.answer_fn(request)
        elif truth is None:
            grids = [[[0]]]
        else:
            roll = self.rng.random()
            if roll < self.correct_prob:
                grids = [list(map(list, g)) for g in truth]
            elif roll < self.correct_prob + self.malformed_prob:
                grids = None
            else:
                salt = abs(hash((task_id, request.seed))) % 100000
                grids = [self._perturb(g, salt + j) for j, g in enumerate(truth)]

        think = self.thinking_tokens.get(request.reasoning_effort, 1000)
        think = max(64, int(think * self.rng.uniform(0.7, 1.3)))

        if grids is None:
            content = "I could not determine a consistent rule for this puzzle."
        else:
            content = self._render(grids)

        if self.latency:
            time.sleep(self.latency)
        elif self.tok_per_sec:
            time.sleep(min(0.05, think / self.tok_per_sec))

        completion = think + max(8, len(content) // 4)
        self.tokens_emitted += completion
        return ChatResponse(
            content=content,
            reasoning="thinking " * min(think // 2, 50),
            usage=Usage(
                prompt_tokens=int(sum(len(m["content"]) for m in request.messages) / 3.2),
                completion_tokens=completion,
                reasoning_tokens=think,
            ),
            finish_reason="stop",
            elapsed=time.monotonic() - started,
            meta=dict(request.meta),
        )


In [ ]:
%%writefile harness/select.py
"""Candidate pooling and the 2-attempt selection rule.

Scoring is per-test-output exact match with two attempts, so votes are tallied
independently per test index (a sample that only parsed for test 1 still
contributes). attempt_1 is the weighted plurality; attempt_2 is the best candidate
with a *different output shape* whenever one exists, because if the plurality shape
is wrong then every same-shape candidate is wrong too.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

from .grids import Grid, GridKey, from_key, shape, to_key
from .llm import EFFORT_RANK

EFFORT_WEIGHT = {name: 1.0 + 0.5 * rank for name, rank in EFFORT_RANK.items()}
TRUNCATION_PENALTY = 0.3
UNKNOWN_EFFORT_WEIGHT = 1.0


@dataclass
class Candidate:
    grid: Grid
    effort: str = "xhigh"
    order: int = 0
    truncated: bool = False

    @property
    def key(self) -> GridKey:
        return to_key(self.grid)

    @property
    def weight(self) -> float:
        w = EFFORT_WEIGHT.get(self.effort, UNKNOWN_EFFORT_WEIGHT)
        return w * (TRUNCATION_PENALTY if self.truncated else 1.0)

    @property
    def effort_rank(self) -> int:
        return EFFORT_RANK.get(self.effort, -1)


@dataclass
class IndexResult:
    attempt_1: Grid
    attempt_2: Grid
    votes: Dict[GridKey, float] = field(default_factory=dict)
    counts: Dict[GridKey, int] = field(default_factory=dict)
    consensus: float = 0.0
    n_candidates: int = 0
    n_distinct: int = 0
    shape_diverse: bool = False
    used_fallback: bool = False

    def as_dict(self) -> Dict[str, Grid]:
        return {"attempt_1": self.attempt_1, "attempt_2": self.attempt_2}


class CandidatePool:
    """Accumulates candidates for one task across all phases and samples."""

    def __init__(self, n_test: int) -> None:
        if n_test < 1:
            raise ValueError("n_test must be >= 1")
        self.n_test = n_test
        self._buckets: List[List[Candidate]] = [[] for _ in range(n_test)]
        self._counter = 0

    def add(self, index: int, grid: Grid, effort: str = "xhigh", truncated: bool = False) -> None:
        if not 0 <= index < self.n_test:
            raise IndexError(f"test index {index} out of range for n_test={self.n_test}")
        self._buckets[index].append(
            Candidate(grid=grid, effort=effort, order=self._counter, truncated=truncated)
        )
        self._counter += 1

    def add_sample(self, grids: Sequence[Optional[Grid]], effort: str, truncated: bool = False) -> int:
        added = 0
        for i, grid in enumerate(grids):
            if grid is not None:
                self.add(i, grid, effort=effort, truncated=truncated)
                added += 1
        return added

    def candidates(self, index: int) -> List[Candidate]:
        return list(self._buckets[index])

    def total(self) -> int:
        return sum(len(b) for b in self._buckets)

    def min_index_count(self) -> int:
        return min(len(b) for b in self._buckets)

    def consensus(self, index: int) -> float:
        if not self._buckets[index]:
            return 0.0
        votes = _tally(self._buckets[index])
        top = max(votes.values())
        return top / sum(votes.values())

    def select(self, fallbacks: Sequence[Grid]) -> List[IndexResult]:
        if len(fallbacks) != self.n_test:
            raise ValueError(f"expected {self.n_test} fallback grids, got {len(fallbacks)}")
        return [_select_index(self._buckets[i], fallbacks[i]) for i in range(self.n_test)]


def _tally(cands: Sequence[Candidate]) -> Dict[GridKey, float]:
    votes: Dict[GridKey, float] = {}
    for c in cands:
        votes[c.key] = votes.get(c.key, 0.0) + c.weight
    return votes


def _rank_key(item: Tuple[GridKey, float], cands: Sequence[Candidate]) -> Tuple[float, int, int]:
    key, weight = item
    best_rank = max(c.effort_rank for c in cands if c.key == key)
    first_order = min(c.order for c in cands if c.key == key)
    return (-weight, -best_rank, first_order)


def _select_index(cands: Sequence[Candidate], fallback: Grid) -> IndexResult:
    if not cands:
        return IndexResult(
            attempt_1=[list(r) for r in fallback],
            attempt_2=[list(r) for r in fallback],
            used_fallback=True,
        )

    votes = _tally(cands)
    counts: Dict[GridKey, int] = {}
    for c in cands:
        counts[c.key] = counts.get(c.key, 0) + 1
    ordered = sorted(votes.items(), key=lambda kv: _rank_key(kv, cands))

    top_key = ordered[0][0]
    top_shape = shape(from_key(top_key))

    diverse = next((k for k, _ in ordered[1:] if shape(from_key(k)) != top_shape), None)
    second_key = diverse if diverse is not None else (ordered[1][0] if len(ordered) > 1 else top_key)

    total_weight = sum(votes.values())
    return IndexResult(
        attempt_1=from_key(top_key),
        attempt_2=from_key(second_key),
        votes=dict(votes),
        counts=counts,
        consensus=(votes[top_key] / total_weight) if total_weight else 0.0,
        n_candidates=len(cands),
        n_distinct=len(votes),
        shape_diverse=diverse is not None,
    )


def consensus_reached(result: IndexResult, min_candidates: int = 4, threshold: float = 0.7) -> bool:
    if result.n_candidates < min_candidates:
        return False
    return result.consensus >= threshold


def task_consensus(results: Sequence[IndexResult], min_candidates: int, threshold: float) -> bool:
    """True when every test index of the task has a confident plurality."""
    if not results:
        return False
    return all(consensus_reached(r, min_candidates, threshold) for r in results)


In [ ]:
%%writefile harness/budget.py
"""Wall-clock and token budget accounting.

The 12h Kaggle rerun is a hard deadline, so the scheduler never trusts its own
throughput forecast: it plans against a safety-discounted horizon, measures the
achieved aggregate decode rate, and re-plans before every wave. All time reads go
through an injectable `Clock` so the governor is unit-testable in milliseconds.
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple


class Clock:
    def now(self) -> float:
        raise NotImplementedError

    def sleep(self, seconds: float) -> None:
        raise NotImplementedError


class SystemClock(Clock):
    def now(self) -> float:
        return time.monotonic()

    def sleep(self, seconds: float) -> None:
        if seconds > 0:
            time.sleep(seconds)


class ManualClock(Clock):
    """Test clock: advance it explicitly instead of waiting."""

    def __init__(self, start: float = 0.0) -> None:
        self._t = float(start)

    def now(self) -> float:
        return self._t

    def sleep(self, seconds: float) -> None:
        self._t += max(0.0, seconds)

    def advance(self, seconds: float) -> None:
        self._t += max(0.0, seconds)

    def set(self, value: float) -> None:
        self._t = float(value)


@dataclass
class BudgetConfig:
    total_seconds: float = 43200.0
    submission_buffer_seconds: float = 600.0
    startup_seconds: float = 1020.0
    finalize_reserve_seconds: float = 120.0
    safety_reserve_fraction: float = 0.10
    planned_tok_per_s: float = 450.0
    pessimistic_tok_per_s: float = 280.0
    optimistic_tok_per_s: float = 700.0
    min_measured_window_s: float = 30.0
    min_measured_tokens: float = 20000.0
    xhigh_tokens_per_sample: int = 10000
    medium_tokens_per_sample: int = 4000
    low_tokens_per_sample: int = 1300

    def tokens_per_sample(self, effort: str) -> int:
        return {
            "xhigh": self.xhigh_tokens_per_sample,
            "medium": self.medium_tokens_per_sample,
            "low": self.low_tokens_per_sample,
        }.get(effort, self.xhigh_tokens_per_sample)


@dataclass
class TokenLedger:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    reasoning_tokens: int = 0
    requests: int = 0
    failures: int = 0
    by_phase: Dict[str, Dict[str, int]] = field(default_factory=dict)
    by_effort: Dict[str, Dict[str, int]] = field(default_factory=dict)

    def record(
        self,
        prompt_tokens: int,
        completion_tokens: int,
        reasoning_tokens: int = 0,
        phase: str = "?",
        effort: str = "?",
        failed: bool = False,
    ) -> None:
        self.requests += 1
        if failed:
            self.failures += 1
        self.prompt_tokens += prompt_tokens
        self.completion_tokens += completion_tokens
        self.reasoning_tokens += reasoning_tokens
        for bucket, key in ((self.by_phase, phase), (self.by_effort, effort)):
            entry = bucket.setdefault(key, {"prompt": 0, "completion": 0, "requests": 0})
            entry["prompt"] += prompt_tokens
            entry["completion"] += completion_tokens
            entry["requests"] += 1

    @property
    def total(self) -> int:
        return self.prompt_tokens + self.completion_tokens

    def as_dict(self) -> Dict[str, object]:
        return {
            "prompt_tokens": self.prompt_tokens,
            "completion_tokens": self.completion_tokens,
            "reasoning_tokens": self.reasoning_tokens,
            "requests": self.requests,
            "failures": self.failures,
            "by_phase": self.by_phase,
            "by_effort": self.by_effort,
        }


class BudgetGovernor:
    def __init__(self, config: Optional[BudgetConfig] = None, clock: Optional[Clock] = None) -> None:
        self.config = config or BudgetConfig()
        self.clock = clock or SystemClock()
        self.ledger = TokenLedger()
        self.t0: float = self.clock.now()
        c = self.config
        self.t_hard = self.t0 + c.total_seconds
        self.t_submit = self.t_hard - c.submission_buffer_seconds
        self.t_gen = self.t_submit - c.finalize_reserve_seconds
        self.t_gen_start = self.t0 + c.startup_seconds
        span = max(1.0, self.t_gen - self.t_gen_start)
        self.t_plan = self.t_gen - c.safety_reserve_fraction * span
        self._gen_tokens_at_start = 0

    @property
    def elapsed(self) -> float:
        return self.clock.now() - self.t0

    def seconds_to(self, horizon: float) -> float:
        return max(0.0, horizon - self.clock.now())

    def plan_seconds_left(self) -> float:
        return self.seconds_to(self.t_plan)

    def gen_seconds_left(self) -> float:
        return self.seconds_to(self.t_gen)

    def hard_seconds_left(self) -> float:
        return self.seconds_to(self.t_submit)

    def must_stop_generating(self) -> bool:
        return self.clock.now() >= self.t_gen

    def must_finalize_now(self) -> bool:
        return self.clock.now() >= self.t_submit - self.config.finalize_reserve_seconds

    def startup_budget_exceeded(self) -> bool:
        return self.clock.now() > self.t_gen_start

    def record(
        self,
        prompt_tokens: int,
        completion_tokens: int,
        reasoning_tokens: int = 0,
        phase: str = "?",
        effort: str = "?",
        failed: bool = False,
    ) -> None:
        self.ledger.record(
            prompt_tokens, completion_tokens, reasoning_tokens, phase, effort, failed
        )

    @property
    def completion_tokens(self) -> int:
        return self.ledger.completion_tokens

    def generation_seconds(self) -> float:
        return max(0.0, self.clock.now() - self.t_gen_start)

    def measured_tok_per_s(self) -> Optional[float]:
        secs = self.generation_seconds()
        tokens = self.ledger.completion_tokens - self._gen_tokens_at_start
        if secs < self.config.min_measured_window_s or tokens < self.config.min_measured_tokens:
            return None
        return tokens / secs

    def projected_tok_per_s(self) -> float:
        measured = self.measured_tok_per_s()
        if measured is None:
            return self.config.planned_tok_per_s
        return max(self.config.pessimistic_tok_per_s * 0.5, measured)

    def affordable_completion_tokens(self, horizon: Optional[float] = None) -> float:
        target = self.t_plan if horizon is None else horizon
        return max(0.0, self.seconds_to(target) * self.projected_tok_per_s())

    def estimate_wave_tokens(self, n_tasks: int, samples_per_task: int, effort: str) -> float:
        return n_tasks * samples_per_task * self.config.tokens_per_sample(effort)

    def plan_wave(
        self, n_tasks: int, samples_per_task: int, effort: str
    ) -> Tuple[int, str]:
        """Clip a proposed wave to what the remaining budget can actually pay for."""
        if n_tasks <= 0 or samples_per_task <= 0:
            return 0, "empty-wave"
        if self.must_stop_generating():
            return 0, "generation-deadline-passed"
        per_sample = self.config.tokens_per_sample(effort)
        remaining = self.affordable_completion_tokens()
        affordable_samples = int(remaining // max(1, n_tasks * per_sample))
        allowed = max(0, min(samples_per_task, affordable_samples))
        if allowed == 0:
            return 0, (
                f"token-budget-exhausted (remaining={remaining:.0f}, "
                f"need={n_tasks * per_sample} for 1 sample/task)"
            )
        if allowed < samples_per_task:
            return allowed, f"trimmed {samples_per_task}->{allowed} to fit token budget"
        return allowed, "full"

    def seconds_for_tokens(self, tokens: float) -> float:
        tps = self.projected_tok_per_s()
        return tokens / tps if tps > 0 else float("inf")

    def snapshot(self) -> Dict[str, object]:
        return {
            "elapsed_s": round(self.elapsed, 1),
            "plan_seconds_left": round(self.plan_seconds_left(), 1),
            "gen_seconds_left": round(self.gen_seconds_left(), 1),
            "hard_seconds_left": round(self.hard_seconds_left(), 1),
            "completion_tokens": self.ledger.completion_tokens,
            "prompt_tokens": self.ledger.prompt_tokens,
            "requests": self.ledger.requests,
            "failures": self.ledger.failures,
            "measured_tok_per_s": (
                round(self.measured_tok_per_s(), 1)
                if self.measured_tok_per_s() is not None
                else None
            ),
            "projected_tok_per_s": round(self.projected_tok_per_s(), 1),
            "affordable_completion_tokens": round(self.affordable_completion_tokens(), 0),
        }


In [ ]:
%%writefile harness/submission.py
"""Cumulative submission writer with schema validation.

The writer is seeded with the competition's own placeholder grid for every task and
every test index before generation starts, then re-written after every wave. A hard
timeout at any point therefore still leaves a complete, schema-valid
`submission.json` on disk. Writes are atomic (temp file + `os.replace`).
"""

from __future__ import annotations

import json
import os
import tempfile
import threading
import time
from typing import Dict, List, Optional, Sequence

from .grids import Grid
from .tasks import ARCTask

PLACEHOLDER: Grid = [[0, 0], [0, 0]]
ATTEMPT_KEYS = ("attempt_1", "attempt_2")
WRITE_ATTEMPTS = 4
WRITE_RETRY_DELAY = 0.05


def _unlink_quietly(path: str) -> None:
    try:
        if os.path.exists(path):
            os.unlink(path)
    except OSError:
        pass


def _coerce_grid(obj: object) -> Grid:
    if isinstance(obj, list) and obj and all(isinstance(r, list) for r in obj):
        out: Grid = []
        for row in obj:
            cells: List[int] = []
            for cell in row:
                if isinstance(cell, bool):
                    cells.append(int(cell))
                elif isinstance(cell, int):
                    cells.append(int(cell))
                elif isinstance(cell, float) and float(cell).is_integer():
                    cells.append(int(cell))
                else:
                    raise ValueError(f"non-integer cell in grid: {cell!r}")
            out.append(cells)
        return out
    raise ValueError(f"not a grid: {type(obj).__name__}")


def build_entry(grids: Sequence[Grid]) -> Dict[str, Grid]:
    attempt_1 = grids[0] if len(grids) > 0 else PLACEHOLDER
    attempt_2 = grids[1] if len(grids) > 1 else attempt_1
    return {"attempt_1": [list(r) for r in attempt_1], "attempt_2": [list(r) for r in attempt_2]}


def empty_submission(tasks: Sequence[ARCTask]) -> Dict[str, List[Dict[str, Grid]]]:
    return {
        task.task_id: [build_entry([PLACEHOLDER]) for _ in range(task.n_test)] for task in tasks
    }


class SubmissionWriter:
    def __init__(self, path: str, tasks: Sequence[ARCTask]) -> None:
        self.path = path
        self.tasks = {t.task_id: t for t in tasks}
        self.order = [t.task_id for t in tasks]
        self.data: Dict[str, List[Dict[str, Grid]]] = empty_submission(tasks)
        self.solved: Dict[str, int] = {}
        self.writes = 0
        self._lock = threading.RLock()
        parent = os.path.dirname(os.path.abspath(path))
        if parent:
            os.makedirs(parent, exist_ok=True)

    def update(self, task_id: str, predictions: Sequence[Dict[str, Grid]]) -> None:
        task = self.tasks.get(task_id)
        if task is None:
            raise KeyError(f"unknown task_id {task_id!r}")
        if len(predictions) != task.n_test:
            raise ValueError(
                f"{task_id}: expected {task.n_test} predictions, got {len(predictions)}"
            )
        entry: List[Dict[str, Grid]] = []
        for pred in predictions:
            present = [k for k in ATTEMPT_KEYS if pred.get(k) is not None]
            grids = [_coerce_grid(pred[k]) for k in present]
            entry.append(build_entry(grids))
        with self._lock:
            self.data[task_id] = entry

    def write(self) -> None:
        """Atomically snapshot the submission to disk.

        Guarded by a re-entrant lock because the watchdog thread and the scheduler
        thread both write. Without it, concurrent `os.replace` raises PermissionError
        on Windows, and `json.dump` iterating `self.data` while `update` mutates it
        raises "dictionary changed size during iteration" on Linux. Losing the write
        at the generation deadline would cost the entire run, so the replace is also
        retried briefly to ride out transient filesystem locks.
        """
        directory = os.path.dirname(os.path.abspath(self.path)) or "."
        os.makedirs(directory, exist_ok=True)
        with self._lock:
            snapshot = json.dumps(self.data)
            last_error: Optional[BaseException] = None
            for attempt in range(WRITE_ATTEMPTS):
                fd, tmp = tempfile.mkstemp(dir=directory, prefix=".submission-", suffix=".tmp")
                try:
                    with os.fdopen(fd, "w", encoding="utf-8") as fh:
                        fh.write(snapshot)
                    os.replace(tmp, self.path)
                    self.writes += 1
                    return
                except PermissionError as exc:
                    last_error = exc
                    _unlink_quietly(tmp)
                    time.sleep(WRITE_RETRY_DELAY * (attempt + 1))
                except BaseException:
                    _unlink_quietly(tmp)
                    raise
            raise last_error if last_error else RuntimeError("submission write failed")

    def validate(self) -> List[str]:
        with self._lock:
            return _validate_data(self.data, self.tasks, self.order)

    def is_complete(self) -> bool:
        return not self.validate()

    @staticmethod
    def load(path: str) -> Dict[str, List[Dict[str, Grid]]]:
        with open(path, "r", encoding="utf-8") as fh:
            return json.load(fh)


def _validate_data(
    data: Dict[str, List[Dict[str, Grid]]],
    tasks: Dict[str, ARCTask],
    order: Sequence[str],
) -> List[str]:
    errors: List[str] = []
    for task_id in order:
        task = tasks[task_id]
        entries = data.get(task_id)
        if entries is None:
            errors.append(f"{task_id}: missing")
            continue
        if len(entries) != task.n_test:
            errors.append(f"{task_id}: {len(entries)} entries, expected {task.n_test}")
            continue
        for i, entry in enumerate(entries):
            if not isinstance(entry, dict) or set(entry.keys()) != set(ATTEMPT_KEYS):
                errors.append(f"{task_id}[{i}]: keys {sorted(entry) if isinstance(entry, dict) else type(entry).__name__}")
                continue
            for key in ATTEMPT_KEYS:
                grid = entry[key]
                if not isinstance(grid, list) or not grid:
                    errors.append(f"{task_id}[{i}].{key}: empty")
                    continue
                if not all(isinstance(r, list) and r for r in grid):
                    errors.append(f"{task_id}[{i}].{key}: ragged or empty rows")
                    continue
                width = len(grid[0])
                if any(len(r) != width for r in grid):
                    errors.append(f"{task_id}[{i}].{key}: non-rectangular")
                    continue
                if any(
                    not isinstance(c, int) or isinstance(c, bool) or not 0 <= c <= 9
                    for r in grid
                    for c in r
                ):
                    errors.append(f"{task_id}[{i}].{key}: cell outside 0..9")
    extra = set(data) - set(order)
    if extra:
        errors.append(f"unexpected task_ids: {sorted(extra)[:5]}")
    return errors


def validate_submission_obj(
    submission: Dict[str, List[Dict[str, Grid]]], tasks: Sequence[ARCTask]
) -> List[str]:
    return _validate_data(
        submission, {t.task_id: t for t in tasks}, [t.task_id for t in tasks]
    )


In [ ]:
%%writefile harness/scoring.py
"""Exact-match scoring against ground truth, mirroring the competition metric.

Score per test output is 1.0 if either attempt matches the solution exactly
(dimensions and every cell), else 0.0. Final score is the mean over all test
outputs, not over tasks.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

from .grids import Grid, grids_equal
from .submission import SubmissionWriter, validate_submission_obj
from .tasks import ARCTask


@dataclass
class TaskScore:
    task_id: str
    n_outputs: int
    correct: int
    attempt_1_correct: int = 0
    attempt_2_only_correct: int = 0

    @property
    def score(self) -> float:
        return self.correct / self.n_outputs if self.n_outputs else 0.0


@dataclass
class ScoreReport:
    tasks: List[TaskScore] = field(default_factory=list)
    total_outputs: int = 0
    total_correct: int = 0
    attempt_1_correct: int = 0
    attempt_2_only_correct: int = 0
    unscored: int = 0

    @property
    def score(self) -> float:
        return self.total_correct / self.total_outputs if self.total_outputs else 0.0

    @property
    def attempt_1_score(self) -> float:
        return self.attempt_1_correct / self.total_outputs if self.total_outputs else 0.0

    @property
    def second_attempt_gain(self) -> float:
        return self.score - self.attempt_1_score

    def solved_task_ids(self) -> List[str]:
        return [t.task_id for t in self.tasks if t.correct == t.n_outputs]

    def failed_task_ids(self) -> List[str]:
        return [t.task_id for t in self.tasks if t.correct == 0]

    def as_dict(self) -> Dict[str, object]:
        return {
            "score": round(self.score, 6),
            "attempt_1_score": round(self.attempt_1_score, 6),
            "second_attempt_gain": round(self.second_attempt_gain, 6),
            "total_outputs": self.total_outputs,
            "total_correct": self.total_correct,
            "attempt_1_correct": self.attempt_1_correct,
            "attempt_2_only_correct": self.attempt_2_only_correct,
            "unscored_tasks": self.unscored,
            "n_tasks_scored": len(self.tasks),
        }


def score_predictions(
    predictions: Sequence[Dict[str, Grid]], ground_truth: Sequence[Grid]
) -> Tuple[int, int, int]:
    correct = a1 = a2_only = 0
    for i, truth in enumerate(ground_truth):
        if i >= len(predictions):
            break
        entry = predictions[i]
        hit_1 = grids_equal(entry.get("attempt_1", []), truth)
        hit_2 = grids_equal(entry.get("attempt_2", []), truth)
        if hit_1:
            a1 += 1
            correct += 1
        elif hit_2:
            a2_only += 1
            correct += 1
    return correct, a1, a2_only


def score_submission(
    submission: Dict[str, List[Dict[str, Grid]]], tasks: Sequence[ARCTask]
) -> ScoreReport:
    report = ScoreReport()
    for task in tasks:
        if task.solutions is None:
            report.unscored += 1
            continue
        preds = submission.get(task.task_id)
        if preds is None:
            report.unscored += 1
            continue
        correct, a1, a2_only = score_predictions(preds, task.solutions)
        report.tasks.append(
            TaskScore(
                task_id=task.task_id,
                n_outputs=task.n_test,
                correct=correct,
                attempt_1_correct=a1,
                attempt_2_only_correct=a2_only,
            )
        )
        report.total_outputs += task.n_test
        report.total_correct += correct
        report.attempt_1_correct += a1
        report.attempt_2_only_correct += a2_only
    return report


def validate_and_score(path: str, tasks: Sequence[ARCTask]) -> Tuple[List[str], Optional[ScoreReport]]:
    submission = SubmissionWriter.load(path)
    errors = validate_submission_obj(submission, tasks)
    report = score_submission(submission, tasks) if not errors else None
    return errors, report


In [ ]:
%%writefile harness/diagnostics.py
"""Coverage instrumentation: the instrument that actually tests hypothesis H1.

The achieved score decomposes into two independent factors:

    score = coverage x selection_efficiency

* `coverage` (pass@N) is the fraction of test outputs for which at least one
  sampled candidate equals the ground truth. It is a hard ceiling set purely by
  model capability plus sample count. No amount of harness work can exceed it.
* `selection_efficiency` is score / coverage: how much of that ceiling the
  plurality + shape-diverse two-attempt rule actually captures.
* `pass_at_1` is the per-sample success rate, the single number that predicts
  coverage and therefore the number worth measuring first on a fresh model.

If coverage on the eval split is below 0.50, H1 ("pure reasoning reaches >=50%") is
refuted no matter what the selection rule does, and the honest conclusion is that
the binding constraint is per-sample capability rather than harness design.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

from .grids import Grid, to_key
from .tasks import ARCTask


@dataclass
class CoverageStats:
    samples: int = 0
    correct_samples: int = 0
    outputs: int = 0
    outputs_covered: int = 0
    tasks: int = 0
    tasks_fully_covered: int = 0
    score: float = 0.0
    by_effort: Dict[str, Dict[str, int]] = field(default_factory=dict)

    @property
    def pass_at_1(self) -> float:
        return self.correct_samples / self.samples if self.samples else 0.0

    @property
    def coverage(self) -> float:
        return self.outputs_covered / self.outputs if self.outputs else 0.0

    @property
    def selection_efficiency(self) -> float:
        return self.score / self.coverage if self.coverage else 0.0

    @property
    def uncovered_outputs(self) -> int:
        return self.outputs - self.outputs_covered

    def as_dict(self) -> Dict[str, object]:
        return {
            "samples": self.samples,
            "correct_samples": self.correct_samples,
            "pass_at_1": round(self.pass_at_1, 5),
            "outputs": self.outputs,
            "outputs_covered": self.outputs_covered,
            "coverage_pass_at_n": round(self.coverage, 5),
            "uncovered_outputs": self.uncovered_outputs,
            "tasks": self.tasks,
            "tasks_fully_covered": self.tasks_fully_covered,
            "score": round(self.score, 5),
            "selection_efficiency": round(self.selection_efficiency, 5),
            "by_effort": self.by_effort,
            "verdict": self.verdict(),
        }

    def verdict(self) -> str:
        if not self.outputs:
            return "no ground truth"
        if self.coverage >= 0.5:
            return "H1 ALIVE: coverage ceiling >= 0.50, score is selection-limited"
        if self.coverage >= 0.4:
            return "H1 MARGINAL: coverage ceiling 0.40-0.50, needs near-perfect selection"
        return "H1 REFUTED: coverage ceiling < 0.40, unreachable by any selection rule"


class CoverageTracker:
    """Records per-sample correctness against ground truth as candidates arrive."""

    def __init__(self, tasks: Sequence[ARCTask]) -> None:
        self._truth: Dict[str, Tuple[Tuple[Tuple[int, ...], ...], ...]] = {}
        self._covered: Dict[str, List[bool]] = {}
        self.stats = CoverageStats()
        for task in tasks:
            if task.solutions is None:
                continue
            self._truth[task.task_id] = tuple(to_key(g) for g in task.solutions)
            self._covered[task.task_id] = [False] * task.n_test
        self.stats.tasks = len(self._truth)
        self.stats.outputs = sum(len(v) for v in self._covered.values())

    @property
    def enabled(self) -> bool:
        return bool(self._truth)

    def observe(self, task_id: str, test_index: int, grid: Grid, effort: str = "?") -> bool:
        truth = self._truth.get(task_id)
        if truth is None:
            return False
        self.stats.samples += 1
        bucket = self.stats.by_effort.setdefault(effort, {"samples": 0, "correct": 0})
        bucket["samples"] += 1
        hit = test_index < len(truth) and to_key(grid) == truth[test_index]
        if hit:
            self.stats.correct_samples += 1
            bucket["correct"] += 1
            flags = self._covered[task_id]
            if test_index < len(flags):
                flags[test_index] = True
        return hit

    def finalize(self, score: Optional[float] = None) -> CoverageStats:
        self.stats.outputs_covered = sum(
            1 for flags in self._covered.values() for f in flags if f
        )
        self.stats.tasks_fully_covered = sum(
            1 for flags in self._covered.values() if flags and all(flags)
        )
        if score is not None:
            self.stats.score = score
        return self.stats


In [ ]:
%%writefile harness/config.py
"""All tunable knobs in one place, with defaults derived from the budget model."""

from __future__ import annotations

import os
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, Optional, Tuple

from .budget import BudgetConfig


@dataclass
class SamplingConfig:
    phase_a_effort: str = "low"
    phase_a_samples: int = 2
    phase_a_max_tokens: int = 2560
    phase_a_temperature: float = 0.7

    phase_b_effort: str = "xhigh"
    wave_samples: Tuple[int, ...] = (3, 3, 3, 3)
    max_xhigh_samples: int = 12
    xhigh_max_tokens: int = 16384
    xhigh_temperature: float = 0.9
    top_p: float = 0.95

    consensus_threshold: float = 0.7
    consensus_min_candidates: int = 4

    concurrency: int = 48
    shuffle_train: bool = False
    request_timeout: float = 2400.0

    def max_tokens_for(self, effort: str) -> int:
        if effort == self.phase_b_effort:
            return self.xhigh_max_tokens
        if effort == "medium":
            return 6144
        return self.phase_a_max_tokens

    def temperature_for(self, effort: str) -> float:
        if effort == self.phase_a_effort:
            return self.phase_a_temperature
        return self.xhigh_temperature


@dataclass
class ServingConfig:
    model: str = "qwen3.8-27b"
    base_url: str = "http://127.0.0.1:8000/v1"
    api_key: str = "EMPTY"
    effort_mechanism: str = "chat_template_kwargs"
    max_retries: int = 3
    health_wait_seconds: float = 900.0
    extra_payload: Dict[str, Any] = field(default_factory=dict)


@dataclass
class HarnessConfig:
    data_dir: Optional[str] = None
    split: str = "test"
    output_dir: Optional[str] = None
    submission_name: str = "submission.json"
    limit: Optional[int] = None
    task_ids: Optional[Tuple[str, ...]] = None
    seed: int = 20260927

    serving: ServingConfig = field(default_factory=ServingConfig)
    sampling: SamplingConfig = field(default_factory=SamplingConfig)
    budget: BudgetConfig = field(default_factory=BudgetConfig)

    log_every: int = 10
    write_after_each_task: bool = False

    @property
    def submission_path(self) -> str:
        from .tasks import resolve_output_dir

        return os.path.join(resolve_output_dir(self.output_dir), self.submission_name)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


In [ ]:
%%writefile harness/watchdog.py
"""Hard watchdog: guarantees a valid submission.json exists before the buffer.

Runs in a daemon thread. At the generation deadline it force-writes whatever the
scheduler has accumulated; at the submission deadline it writes once more and, if
`hard_exit` is set, terminates the process so a wedged vLLM client or a hung CUDA
context cannot cost the whole run.
"""

from __future__ import annotations

import os
import threading
import time
from typing import Callable, Optional

from .budget import BudgetGovernor
from .submission import SubmissionWriter

Logger = Callable[[str], None]


class Watchdog:
    def __init__(
        self,
        governor: BudgetGovernor,
        writer: SubmissionWriter,
        logger: Optional[Logger] = None,
        hard_exit: bool = False,
        poll_seconds: float = 5.0,
        clock: Optional[Callable[[], float]] = None,
        sleeper: Optional[Callable[[float], None]] = None,
    ) -> None:
        self.governor = governor
        self.writer = writer
        self.log: Logger = logger or (lambda _m: None)
        self.hard_exit = hard_exit
        self.poll_seconds = poll_seconds
        self._now = clock or time.monotonic
        self._sleep = sleeper or time.sleep
        self._thread: Optional[threading.Thread] = None
        self._stop = threading.Event()
        self.finalized = False
        self.triggered_at: Optional[float] = None
        self.errors: list = []

    def start(self) -> None:
        if self._thread is not None:
            return
        self._thread = threading.Thread(target=self._loop, name="arc-watchdog", daemon=True)
        self._thread.start()

    def stop(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.poll_seconds * 2 + 1)

    def _safe_write(self) -> bool:
        try:
            errors = self.writer.validate()
            if errors:
                self.errors.extend(errors[:10])
                self.log(f"watchdog: submission has {len(errors)} schema errors (writing anyway)")
            self.writer.write()
            return True
        except Exception as exc:
            self.errors.append(f"{type(exc).__name__}: {exc}")
            self.log(f"watchdog: write failed: {exc}")
            return False

    def _loop(self) -> None:
        while not self._stop.is_set():
            now = self._now()
            if not self.finalized and now >= self.governor.t_gen:
                self.triggered_at = now
                self.log(f"watchdog: generation deadline hit at t+{now - self.governor.t0:.0f}s, finalizing")
                self._safe_write()
                self.finalized = True
            if now >= self.governor.t_submit - self.governor.config.finalize_reserve_seconds:
                self._safe_write()
                if self.hard_exit:
                    self.log("watchdog: submission buffer reached, hard exit")
                    self._sleep(1.0)
                    os._exit(0)
                self.log("watchdog: final submission written, nothing left to guard")
                break
            self._sleep(self.poll_seconds)

    def tick(self) -> None:
        """Synchronous equivalent of one watchdog loop iteration (used by tests)."""
        now = self._now()
        if not self.finalized and now >= self.governor.t_gen:
            self.triggered_at = now
            self.log(f"watchdog: generation deadline hit at t+{now - self.governor.t0:.0f}s, finalizing")
            self._safe_write()
            self.finalized = True


In [ ]:
%%writefile harness/waves.py
"""Wave scheduler: cheap triage pass over every task, then budget-routed xhigh waves.

Design consequences of the budget model in HYPOTHESIS.md:

* Phase A runs `low`-effort samples over **all** tasks first. It costs ~4% of the
  token budget and ~25 minutes, and after it completes every task already has a
  schema-valid two-attempt answer on disk. That converts a 12-hour all-or-nothing
  run into a run whose worst case is "phase A quality", not zero.
* Phase B runs in *waves* rather than task-by-task. A single task only needs ~7
  concurrent xhigh samples, and a batch of 7 decodes at ~220 tok/s against a
  ~760 tok/s ceiling at batch 32. Waves over all tasks keep the in-flight pool at
  `concurrency` requests, which is what actually buys the throughput the budget
  assumes.
* Before every wave the governor re-plans against *measured* throughput, so a 2x
  throughput miss degrades samples-per-task instead of blowing the deadline.
"""

from __future__ import annotations

import random
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout, as_completed
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence, Tuple

from .budget import BudgetGovernor
from .config import HarnessConfig
from .diagnostics import CoverageTracker
from .grids import Grid, fallback_grid
from .llm import ChatRequest, LLMClient
from .parse import merge_reasoning_and_content, parse_answers
from .prompts import build_messages
from .select import CandidatePool, IndexResult, task_consensus
from .submission import SubmissionWriter
from .tasks import ARCTask, total_test_outputs

Logger = Callable[[str], None]


@dataclass
class _Job:
    request: ChatRequest
    state: "TaskState"
    phase: str
    effort: str


@dataclass
class TaskState:
    task: ARCTask
    pool: CandidatePool
    base_seed: int
    results: List[IndexResult] = field(default_factory=list)
    fallbacks: List[Grid] = field(default_factory=list)
    stopped: bool = False
    samples_by_effort: Dict[str, int] = field(default_factory=dict)
    parse_methods: Dict[str, int] = field(default_factory=dict)
    failures: int = 0
    seconds: float = 0.0

    @property
    def n_samples(self) -> int:
        return sum(self.samples_by_effort.values())

    def xhigh_samples(self) -> int:
        return self.samples_by_effort.get("xhigh", 0)


@dataclass
class WaveRecord:
    index: int
    phase: str
    effort: str
    tasks: int
    requests: int
    samples_per_task: int
    completion_tokens: int = 0
    seconds: float = 0.0
    trimmed_reason: str = ""
    stopped_early: bool = False
    consensus_after: int = 0


@dataclass
class RunReport:
    n_tasks: int = 0
    n_test_outputs: int = 0
    elapsed_s: float = 0.0
    waves: List[WaveRecord] = field(default_factory=list)
    samples_per_task: float = 0.0
    xhigh_samples_per_task: float = 0.0
    consensus_stopped_tasks: int = 0
    tasks_with_no_parse: int = 0
    request_failures: int = 0
    parse_method_counts: Dict[str, int] = field(default_factory=dict)
    measured_tok_per_s: Optional[float] = None
    governor: Dict[str, object] = field(default_factory=dict)
    tokens: Dict[str, object] = field(default_factory=dict)
    score: Optional[Dict[str, object]] = None
    coverage: Optional[Dict[str, object]] = None
    submission_errors: List[str] = field(default_factory=list)
    submission_path: str = ""

    def as_dict(self) -> Dict[str, object]:
        return {
            "n_tasks": self.n_tasks,
            "n_test_outputs": self.n_test_outputs,
            "elapsed_s": round(self.elapsed_s, 1),
            "samples_per_task": round(self.samples_per_task, 2),
            "xhigh_samples_per_task": round(self.xhigh_samples_per_task, 2),
            "consensus_stopped_tasks": self.consensus_stopped_tasks,
            "tasks_with_no_parse": self.tasks_with_no_parse,
            "request_failures": self.request_failures,
            "parse_method_counts": self.parse_method_counts,
            "measured_tok_per_s": self.measured_tok_per_s,
            "governor": self.governor,
            "tokens": self.tokens,
            "score": self.score,
            "coverage": self.coverage,
            "submission_errors": self.submission_errors,
            "submission_path": self.submission_path,
            "waves": [w.__dict__ for w in self.waves],
        }


class Scheduler:
    def __init__(
        self,
        client: LLMClient,
        tasks: Sequence[ARCTask],
        config: HarnessConfig,
        governor: BudgetGovernor,
        writer: SubmissionWriter,
        logger: Optional[Logger] = None,
        tracker: Optional["CoverageTracker"] = None,
    ) -> None:
        self.client = client
        self.tasks = list(tasks)
        self.config = config
        self.governor = governor
        self.writer = writer
        self.tracker = tracker
        self.log: Logger = logger or (lambda _m: None)
        rng = random.Random(config.seed)
        order = list(self.tasks)
        rng.shuffle(order)
        self.states: Dict[str, TaskState] = {}
        for i, task in enumerate(order):
            state = TaskState(
                task=task,
                pool=CandidatePool(task.n_test),
                base_seed=rng.randrange(1, 2**30),
                fallbacks=[fallback_grid(task.test[j], task.train) for j in range(task.n_test)],
            )
            state.results = state.pool.select(state.fallbacks)
            self.states[task.task_id] = state
        self.order: List[str] = [t.task_id for t in order]
        self.report = RunReport()
        self._parse_counts: Dict[str, int] = {}

    def _request(self, state: TaskState, effort: str, sample_idx: int, phase: str) -> ChatRequest:
        sampling = self.config.sampling
        seed = state.base_seed + 1013 * sample_idx
        return ChatRequest(
            messages=build_messages(state.task, shuffle_train=sampling.shuffle_train, seed=seed),
            max_tokens=sampling.max_tokens_for(effort),
            temperature=sampling.temperature_for(effort),
            top_p=sampling.top_p,
            seed=seed,
            reasoning_effort=effort,
            meta={
                "task_id": state.task.task_id,
                "phase": phase,
                "effort": effort,
                "sample": sample_idx,
                "n_test": state.task.n_test,
            },
        )

    def _handle(self, response, job: _Job) -> None:
        state = job.state
        usage = response.usage
        self.governor.record(
            prompt_tokens=usage.prompt_tokens,
            completion_tokens=usage.completion_tokens,
            reasoning_tokens=usage.reasoning_tokens,
            phase=job.phase,
            effort=job.effort,
            failed=not response.ok,
        )
        state.seconds += response.elapsed
        if not response.ok:
            state.failures += 1
            self.log(f"  ! {state.task.task_id} {job.phase}/{job.effort}: {response.error}")
            return
        text = merge_reasoning_and_content(response.reasoning, response.content)
        parsed = parse_answers(text, state.task.n_test)
        self._parse_counts[parsed.method] = self._parse_counts.get(parsed.method, 0) + 1
        state.parse_methods[parsed.method] = state.parse_methods.get(parsed.method, 0) + 1
        grids = parsed.as_list(state.task.n_test)
        if self.tracker is not None and self.tracker.enabled:
            for i, grid in enumerate(grids):
                if grid is not None:
                    self.tracker.observe(state.task.task_id, i, grid, job.effort)
        added = state.pool.add_sample(grids, effort=job.effort, truncated=response.truncated)
        if added:
            state.samples_by_effort[job.effort] = state.samples_by_effort.get(job.effort, 0) + 1
        self._refresh(state)
        if self.config.write_after_each_task:
            self._update_writer(state)

    def _refresh(self, state: TaskState) -> None:
        state.results = state.pool.select(state.fallbacks)
        sampling = self.config.sampling
        state.stopped = task_consensus(
            state.results, sampling.consensus_min_candidates, sampling.consensus_threshold
        )

    def _update_writer(self, state: TaskState) -> None:
        self.writer.update(state.task.task_id, [r.as_dict() for r in state.results])

    def _flush_all(self) -> None:
        for task_id in self.order:
            self._update_writer(self.states[task_id])
        self.writer.write()

    def _eligible(self) -> List[str]:
        return [tid for tid in self.order if not self.states[tid].stopped]

    def _execute(self, jobs: Sequence[_Job], wave_timeout: float) -> Tuple[int, bool]:
        """Run a batch of requests; returns (completed, stopped_early)."""
        if not jobs:
            return 0, False
        workers = max(1, min(self.config.sampling.concurrency, len(jobs)))
        completed = 0
        stopped_early = False
        executor = ThreadPoolExecutor(max_workers=workers, thread_name_prefix="llm")
        futures = {executor.submit(self.client.complete, job.request): job for job in jobs}
        try:
            for future in as_completed(futures, timeout=max(1.0, wave_timeout)):
                job = futures[future]
                try:
                    response = future.result()
                except Exception as exc:
                    response = type(
                        "R",
                        (),
                        {
                            "ok": False,
                            "error": f"{type(exc).__name__}: {exc}",
                            "elapsed": 0.0,
                            "usage": type("U", (), {"prompt_tokens": 0, "completion_tokens": 0, "reasoning_tokens": 0})(),
                            "truncated": False,
                            "content": "",
                            "reasoning": "",
                        },
                    )()
                self._handle(response, job)
                completed += 1
                if completed % self.config.log_every == 0:
                    self.log(
                        f"    {completed}/{len(jobs)} done | "
                        f"completion_tokens={self.governor.completion_tokens} | "
                        f"gen_left={self.governor.gen_seconds_left():.0f}s"
                    )
                if self.governor.must_stop_generating():
                    stopped_early = True
                    break
        except FuturesTimeout:
            stopped_early = True
            self.log(f"    wave timeout after {wave_timeout:.0f}s")
        finally:
            executor.shutdown(wait=False, cancel_futures=True)
        return completed, stopped_early

    def run(self) -> RunReport:
        started = time.monotonic()
        self.report.n_tasks = len(self.tasks)
        self.report.n_test_outputs = total_test_outputs(self.tasks)
        self.log(
            f"scheduler: {self.report.n_tasks} tasks / {self.report.n_test_outputs} test outputs; "
            f"plan horizon {self.governor.plan_seconds_left():.0f}s; "
            f"projected {self.governor.projected_tok_per_s():.0f} tok/s"
        )
        self._flush_all()

        self._run_phase_a()
        self._run_phase_b()

        self._flush_all()
        self.report.elapsed_s = time.monotonic() - started
        self._finalize_report()
        return self.report

    def _run_phase_a(self) -> None:
        sampling = self.config.sampling
        n = sampling.phase_a_samples
        if n <= 0:
            return
        allowed, reason = self.governor.plan_wave(len(self.order), n, sampling.phase_a_effort)
        self.log(f"phase A ({sampling.phase_a_effort}): plan={n}/task -> {allowed} ({reason})")
        if allowed <= 0:
            return
        t0 = time.monotonic()
        before = self.governor.completion_tokens
        jobs: List[_Job] = []
        for task_id in self.order:
            state = self.states[task_id]
            for s in range(allowed):
                jobs.append(
                    _Job(
                        request=self._request(state, sampling.phase_a_effort, s, "A"),
                        state=state,
                        phase="A",
                        effort=sampling.phase_a_effort,
                    )
                )
        wave_timeout = min(
            self.governor.gen_seconds_left(),
            max(60.0, self.governor.seconds_for_tokens(len(jobs) * sampling.phase_a_max_tokens) * 2.0),
        )
        completed, stopped = self._execute(jobs, wave_timeout)
        record = WaveRecord(
            index=0,
            phase="A",
            effort=sampling.phase_a_effort,
            tasks=len(self.order),
            requests=completed,
            samples_per_task=allowed,
            completion_tokens=self.governor.completion_tokens - before,
            seconds=time.monotonic() - t0,
            trimmed_reason=reason if reason != "full" else "",
            stopped_early=stopped,
            consensus_after=sum(1 for s in self.states.values() if s.stopped),
        )
        self.report.waves.append(record)
        self._flush_all()
        self.log(
            f"phase A done: {completed} requests, {record.completion_tokens} completion tokens, "
            f"{record.seconds:.0f}s, consensus on {record.consensus_after} tasks; submission written"
        )

    def _run_phase_b(self) -> None:
        sampling = self.config.sampling
        for wave_idx, requested in enumerate(sampling.wave_samples, start=1):
            if self.governor.must_stop_generating():
                self.log(f"phase B: stopping before wave {wave_idx}, generation deadline reached")
                break
            eligible = self._eligible()
            if not eligible:
                self.log(f"phase B: all {len(self.order)} tasks at consensus after wave {wave_idx - 1}")
                break
            remaining_cap = sampling.max_xhigh_samples
            allowed, reason = self.governor.plan_wave(len(eligible), requested, sampling.phase_b_effort)
            self.log(
                f"phase B wave {wave_idx}: {len(eligible)} eligible tasks, "
                f"plan={requested} -> {allowed} samples/task ({reason}); cap={remaining_cap}"
            )
            if allowed <= 0:
                self.report.waves.append(
                    WaveRecord(
                        index=wave_idx,
                        phase="B",
                        effort=sampling.phase_b_effort,
                        tasks=len(eligible),
                        requests=0,
                        samples_per_task=0,
                        trimmed_reason=reason,
                    )
                )
                break

            t0 = time.monotonic()
            before = self.governor.completion_tokens
            jobs: List[_Job] = []
            for task_id in eligible:
                state = self.states[task_id]
                room = max(0, remaining_cap - state.xhigh_samples())
                for s in range(min(allowed, room)):
                    jobs.append(
                        _Job(
                            request=self._request(
                                state, sampling.phase_b_effort, 100 + state.xhigh_samples() + s, f"B{wave_idx}"
                            ),
                            state=state,
                            phase=f"B{wave_idx}",
                            effort=sampling.phase_b_effort,
                        )
                    )
            est_tokens = len(jobs) * self.governor.config.tokens_per_sample(sampling.phase_b_effort)
            wave_timeout = min(
                self.governor.gen_seconds_left(),
                max(120.0, self.governor.seconds_for_tokens(est_tokens) * 1.8),
            )
            completed, stopped = self._execute(jobs, wave_timeout)
            record = WaveRecord(
                index=wave_idx,
                phase=f"B{wave_idx}",
                effort=sampling.phase_b_effort,
                tasks=len(eligible),
                requests=completed,
                samples_per_task=allowed,
                completion_tokens=self.governor.completion_tokens - before,
                seconds=time.monotonic() - t0,
                trimmed_reason=reason if reason != "full" else "",
                stopped_early=stopped,
                consensus_after=sum(1 for s in self.states.values() if s.stopped),
            )
            self.report.waves.append(record)
            self._flush_all()
            self.log(
                f"phase B wave {wave_idx} done: {completed} requests, "
                f"{record.completion_tokens} completion tokens, {record.seconds:.0f}s, "
                f"consensus on {record.consensus_after}/{len(self.order)} tasks"
            )
            if stopped:
                self.log("phase B: wave stopped early by deadline, finalizing")
                break

    def _finalize_report(self) -> None:
        states = list(self.states.values())
        n = max(1, len(states))
        self.report.samples_per_task = sum(s.n_samples for s in states) / n
        self.report.xhigh_samples_per_task = sum(s.xhigh_samples() for s in states) / n
        self.report.consensus_stopped_tasks = sum(1 for s in states if s.stopped)
        self.report.tasks_with_no_parse = sum(1 for s in states if s.pool.total() == 0)
        self.report.request_failures = self.governor.ledger.failures
        self.report.parse_method_counts = dict(self._parse_counts)
        measured = self.governor.measured_tok_per_s()
        self.report.measured_tok_per_s = round(measured, 1) if measured else None
        self.report.governor = self.governor.snapshot()
        self.report.tokens = self.governor.ledger.as_dict()
        self.report.submission_path = self.writer.path
        self.report.submission_errors = self.writer.validate()

    def per_task_rows(self) -> List[Dict[str, object]]:
        rows = []
        for task_id in sorted(self.states):
            state = self.states[task_id]
            rows.append(
                {
                    "task_id": task_id,
                    "n_test": state.task.n_test,
                    "samples": state.n_samples,
                    "xhigh_samples": state.xhigh_samples(),
                    "candidates": state.pool.total(),
                    "distinct": sum(r.n_distinct for r in state.results),
                    "consensus": round(min((r.consensus for r in state.results), default=0.0), 3),
                    "stopped": state.stopped,
                    "failures": state.failures,
                    "seconds": round(state.seconds, 1),
                    "parse": state.parse_methods,
                }
            )
        return rows


In [ ]:
%%writefile harness/runner.py
"""Top-level orchestration shared by `scripts/eval_local.py` and the Kaggle notebook."""

from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Sequence, Tuple

from .budget import BudgetConfig, BudgetGovernor, Clock, SystemClock
from .config import HarnessConfig
from .diagnostics import CoverageStats, CoverageTracker
from .grids import Grid
from .llm import HTTPChatClient, LLMClient, MockLLM, validate_effort
from .scoring import ScoreReport, score_submission
from .submission import SubmissionWriter
from .tasks import (
    ARCTask,
    load_challenges,
    resolve_challenges_path,
    resolve_data_dir,
    resolve_output_dir,
    resolve_solutions_path,
    total_test_outputs,
)
from .watchdog import Watchdog
from .waves import RunReport, Scheduler

Logger = Callable[[str], None]


def default_logger(prefix: str = "h1") -> Logger:
    def _log(message: str) -> None:
        print(f"[{time.strftime('%H:%M:%S')}] {prefix}: {message}", flush=True)

    return _log


def load_tasks(config: HarnessConfig) -> List[ARCTask]:
    data_dir = resolve_data_dir(config.data_dir)
    challenges = resolve_challenges_path(data_dir, config.split)
    solutions = resolve_solutions_path(data_dir, config.split)
    tasks = load_challenges(challenges, solutions)
    if config.task_ids:
        wanted = set(config.task_ids)
        tasks = [t for t in tasks if t.task_id in wanted]
    if config.limit:
        tasks = tasks[: config.limit]
    return tasks


def build_client(config: HarnessConfig, answers: Optional[Dict[str, Sequence[Grid]]] = None) -> LLMClient:
    if answers is not None:
        return MockLLM(answers=answers)
    return HTTPChatClient(
        base_url=config.serving.base_url,
        model=config.serving.model,
        api_key=config.serving.api_key,
        timeout=config.sampling.request_timeout,
        max_retries=config.serving.max_retries,
        effort_mechanism=config.serving.effort_mechanism,
        extra_payload=config.serving.extra_payload,
    )


def mock_answers(tasks: Sequence[ARCTask]) -> Dict[str, List[Grid]]:
    return {t.task_id: [list(map(list, g)) for g in (t.solutions or [])] for t in tasks}


@dataclass
class RunOutcome:
    report: RunReport
    score: Optional[ScoreReport]
    submission_path: str
    tasks: List[ARCTask]
    config: HarnessConfig

    def write_artifacts(self, output_dir: str) -> Dict[str, str]:
        os.makedirs(output_dir, exist_ok=True)
        paths = {}
        report_path = os.path.join(output_dir, "run_report.json")
        with open(report_path, "w", encoding="utf-8") as fh:
            json.dump(self.report.as_dict(), fh, indent=2, default=str)
        paths["report"] = report_path

        config_path = os.path.join(output_dir, "config.json")
        with open(config_path, "w", encoding="utf-8") as fh:
            json.dump(self.config.to_dict(), fh, indent=2, default=str)
        paths["config"] = config_path

        if self.score is not None:
            score_path = os.path.join(output_dir, "score.json")
            with open(score_path, "w", encoding="utf-8") as fh:
                json.dump(self.score.as_dict(), fh, indent=2)
            paths["score"] = score_path
        return paths


def run_harness(
    config: HarnessConfig,
    client: Optional[LLMClient] = None,
    clock: Optional[Clock] = None,
    logger: Optional[Logger] = None,
    tasks: Optional[Sequence[ARCTask]] = None,
    governor: Optional[BudgetGovernor] = None,
    submission_path: Optional[str] = None,
    use_watchdog: bool = False,
    hard_exit: bool = False,
    watchdog_poll_seconds: float = 5.0,
    watchdog_sleeper: Optional[Callable[[float], None]] = None,
) -> RunOutcome:
    log = logger or default_logger()
    clock = clock or SystemClock()
    tasks = list(tasks) if tasks is not None else load_tasks(config)
    if not tasks:
        raise ValueError("no tasks loaded")

    validate_effort(config.sampling.phase_a_effort)
    validate_effort(config.sampling.phase_b_effort)

    output_dir = resolve_output_dir(config.output_dir)
    os.makedirs(output_dir, exist_ok=True)
    path = submission_path or os.path.join(output_dir, config.submission_name)

    governor = governor or BudgetGovernor(config.budget, clock=clock)
    writer = SubmissionWriter(path, tasks)
    if client is None:
        client = build_client(config, answers=mock_answers(tasks) if config.split == "mock" else None)

    log(
        f"run_harness: {len(tasks)} tasks, {total_test_outputs(tasks)} test outputs, "
        f"split={config.split}, submission={path}"
    )
    tracker = CoverageTracker(tasks)
    watchdog = None
    if use_watchdog:
        watchdog = Watchdog(
            governor,
            writer,
            logger=log,
            hard_exit=hard_exit,
            poll_seconds=watchdog_poll_seconds,
            clock=clock.now,
            sleeper=watchdog_sleeper or clock.sleep,
        )
        watchdog.start()
        log(
            f"watchdog armed: finalize at t+{governor.t_gen - governor.t0:.0f}s, "
            f"hard exit at t+{governor.t_submit - governor.config.finalize_reserve_seconds - governor.t0:.0f}s"
        )
    try:
        scheduler = Scheduler(client, tasks, config, governor, writer, logger=log, tracker=tracker)
        report = scheduler.run()
    finally:
        if watchdog is not None:
            watchdog.stop()
            writer.write()

    score: Optional[ScoreReport] = None
    if tracker.enabled:
        score = score_submission(writer.data, tasks)
        report.score = score.as_dict()
        coverage = tracker.finalize(score.score)
        report.coverage = coverage.as_dict()
        log(
            f"score={score.score * 100:.2f}% "
            f"(attempt_1 only={score.attempt_1_score * 100:.2f}%, "
            f"2nd-attempt gain={score.second_attempt_gain * 100:.2f}%, "
            f"{score.total_correct}/{score.total_outputs} outputs)"
        )
        log(
            f"coverage(pass@N)={coverage.coverage * 100:.2f}% "
            f"pass@1={coverage.pass_at_1 * 100:.2f}% "
            f"selection_efficiency={coverage.selection_efficiency * 100:.1f}% "
            f"-> {coverage.verdict()}"
        )

    errors = report.submission_errors
    if errors:
        log(f"SUBMISSION SCHEMA ERRORS ({len(errors)}): {errors[:5]}")
    else:
        log("submission schema valid")

    outcome = RunOutcome(report=report, score=score, submission_path=path, tasks=tasks, config=config)
    detail_path = os.path.join(output_dir, "per_task_detail.json")
    with open(detail_path, "w", encoding="utf-8") as fh:
        json.dump(scheduler.per_task_rows(), fh, indent=2)
    outcome.write_artifacts(output_dir)
    return outcome


def write_config_snapshot(config: HarnessConfig, path: str) -> None:
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(config.to_dict(), fh, indent=2, default=str)


def budget_table(
    config: Optional[BudgetConfig] = None,
    n_tasks: int = 240,
    phase_a_samples: int = 2,
) -> List[Dict[str, object]]:
    """Reproduce the HYPOTHESIS.md budget arithmetic from live config values."""
    c = config or BudgetConfig()
    usable = (
        c.total_seconds
        - c.submission_buffer_seconds
        - c.startup_seconds
        - c.finalize_reserve_seconds
    )
    planned = usable * (1.0 - c.safety_reserve_fraction)
    phase_a_tokens = n_tasks * phase_a_samples * c.low_tokens_per_sample
    rows = []
    for name, tps in (
        ("pessimistic", c.pessimistic_tok_per_s),
        ("central", c.planned_tok_per_s),
        ("optimistic", c.optimistic_tok_per_s),
    ):
        tokens = planned * tps
        phase_b = max(0.0, tokens - phase_a_tokens)
        rows.append(
            {
                "scenario": name,
                "tok_per_s": tps,
                "usable_seconds": round(usable, 0),
                "planned_seconds": round(planned, 0),
                "completion_token_budget": round(tokens, 0),
                "phase_a_tokens": round(phase_a_tokens, 0),
                "tokens_per_task": round(tokens / max(1, n_tasks), 0),
                "xhigh_samples_per_task_uniform": round(
                    phase_b / max(1, n_tasks) / c.xhigh_tokens_per_sample, 2
                ),
                "xhigh_wave_hours_uniform": round(
                    phase_b / max(1e-9, tps) / 3600.0, 2
                ),
            }
        )
    return rows


In [ ]:
import os
print(sorted(os.listdir('harness')))
import harness
print('harness import OK:', len(harness.__all__), 'exports')


# Compute budget — the numbers this kernel schedules against

Reproduced live by `python scripts/eval_local.py --budget-table`.

**Hardware** 4 × L4: 24 GB, ~300 GB/s, sm_89 Ada, **no NVLink** → TP all-reduce is latency-bound.
**Model** Qwen3.8-27B FP8 ≈ 27 GB → TP=4 → **6.75 GB weights/GPU**.
**KV** only 16 of 64 layers are full attention (4 KV heads, head_dim 256):
`16 × 4 × 256 × 2 × 2 B = 64 KB/token` aggregate = **16 KB/token/rank**.

### Memory (per GPU, TP=4)

```
24.0 × 0.92 util          = 22.08 GB
− 6.75 weight shard       = 15.33 GB
− ~2.0 ctx/activ/DeltaNet = 13.33 GB KV pool
13.33e9 / 16,384 B        = 813,600 tokens/rank
worst seq = 4,255 prompt + 16,384 completion = 20,640 → 39 concurrent
mean  seq = 1,135 prompt + 10,000 completion = 11,135 → 73 concurrent
```
→ `--max-num-seqs 64`, harness `concurrency=48`. Memory is **not** the constraint.

### Decode step (per rank): `22.5 ms weights + B·L̄·16 KB/300 GB/s + ~6 ms collectives`

| B | L̄ | step | raw tok/s | derated ×0.55 | BW/rank |
|---:|---:|---:|---:|---:|---:|
| 8 | 8 K | 31.9 ms | 251 | 138 | 51% |
| 32 | 8 K | 42.1 ms | 760 | 418 | 87% |
| 48 | 11 K | 54.1 ms | 887 | 488 | 95% |

→ **planning figures: 280 (pessimistic) / 450 (central) / 700 (optimistic) tok/s aggregate.**
Cross-check: 450 tok/s = 24.3 TFLOPS = **2.5% MFU** of 968 TFLOPS FP8 peak — decode is
memory-bound, so a low MFU is expected and confirms we are not compute-limited.

### Wall clock

```
43,200 s (12 h) − 600 submission buffer − 120 finalize reserve = 42,480 s  (t_gen)
             − 1,020 s startup/load/compile                     = 41,460 s  usable
             − 10% safety reserve (4,146 s)                     = 37,314 s  planned window
```

### Tokens and the answer to the crux question

| Scenario | tok/s | completion budget | tok/task | **xhigh samples/task** | total run |
|---|---:|---:|---:|---:|---:|
| Pessimistic | 280 | 10.45 M | 43,533 | **4.09** | 10.36 h |
| Central | 450 | 16.79 M | 69,964 | **6.74** | 10.35 h |
| Optimistic | 700 | 26.12 M | 108,832 | **10.62** | 10.37 h |

Prompt cost is measured, not guessed: **test split mean 1,135 tokens** (p90 2,067, max 4,255);
evaluation split mean 1,946 (max 5,379). With prefix caching, total prefill ≈ 76 s = 0.2% of
the window. Worst case 4,255 + 16,384 = 20,639 < 32,768 → `--max-model-len 32768` is safe.

**Phase A** = `240 × 2 × 1,300 = 624,000` tokens = **3.7%** of budget = **23 min**.
**Phase B** = central `(16.79 M − 0.62 M) / 240 / 10,000` = **6.74 xhigh samples/task**.

### Why waves and not task-by-task

Serial scheduling keeps only ~6.7 requests in flight → step 30.5 ms → **230 tok/s** →
16.17 M tokens would take **19.5 h**. ✗ **Serial execution of this exact hypothesis overruns
the budget by ~1.9×.** Waves over all tasks at `concurrency=48` are what make B≈40 and
therefore 450 tok/s reachable. This is a precondition of the hypothesis, not an optimisation.


## 2. Runtime

Configuration, dependency install, checkpoint discovery, vLLM launch ladder, live calibration, the harness run, and final validation.


In [ ]:
"""H1 runtime configuration: paths, deadline, and the knobs derived in HYPOTHESIS.md §4-5."""
import os
import sys
import time

RUN_START_MONO = time.monotonic()
RUN_START_WALL = time.time()

CWD = os.getcwd()
if CWD not in sys.path:
    sys.path.insert(0, CWD)

IS_RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else CWD
INPUT_ROOT = "/kaggle/input"
COMP_DIR = os.path.join(INPUT_ROOT, "competitions", "arc-prize-2026-arc-agi-2")

TOTAL_HOURS = 12.0
SUBMISSION_BUFFER_S = 600.0
FINALIZE_RESERVE_S = 120.0
SAFETY_RESERVE_FRACTION = 0.10
LAUNCH_RESERVE_S = 2700.0

SERVED_MODEL_NAME = "qwen3.8-27b"
VLLM_PORT = 8000
VLLM_PORT_ALT = 8001
BASE_URL = "http://127.0.0.1:%d/v1" % VLLM_PORT
BASE_URL_ALT = "http://127.0.0.1:%d/v1" % VLLM_PORT_ALT
VLLM_LOG = os.path.join(WORKING, "vllm.log")
VLLM_LOG_ALT = os.path.join(WORKING, "vllm_alt.log")

SPLIT = "test" if IS_RERUN else "evaluation"
DATA_DIR = COMP_DIR if os.path.isdir(COMP_DIR) else os.path.join(CWD, "data")
LIMIT = None

PHASE_A_EFFORT = "low"
PHASE_A_SAMPLES = 2
PHASE_A_MAX_TOKENS = 2560
WAVE_SAMPLES = (3, 3, 3, 3)
MAX_XHIGH_SAMPLES = 12
XHIGH_MAX_TOKENS = 16384
XHIGH_TEMPERATURE = 0.9
TOP_P = 0.95
CONCURRENCY = 48
CONSENSUS_THRESHOLD = 0.70
CONSENSUS_MIN_CANDIDATES = 4
SHUFFLE_TRAIN = False

PLANNED_TOK_S = 450.0
PESSIMISTIC_TOK_S = 280.0
OPTIMISTIC_TOK_S = 700.0
XHIGH_TOKENS_PER_SAMPLE = 10000
LOW_TOKENS_PER_SAMPLE = 1300


def log(msg):
    print("[%s] %s" % (time.strftime("%H:%M:%S"), msg), flush=True)


def elapsed_s():
    return time.monotonic() - RUN_START_MONO


def remaining_s():
    return TOTAL_HOURS * 3600.0 - elapsed_s()


log("rerun=%s split=%s" % (IS_RERUN, SPLIT))
log("data_dir=%s" % DATA_DIR)
log("working=%s" % WORKING)
log("remaining=%.2f h" % (remaining_s() / 3600.0))


In [ ]:
"""Install vLLM: pre-baked wheelhouse first (offline rerun), pip second (internet save)."""
import glob
import importlib
import os
import subprocess
import sys


def sh(cmd, timeout=3600):
    log("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, stdout=sys.stdout, stderr=sys.stderr, timeout=timeout)
    return proc.returncode


def find_wheelhouse():
    patterns = [
        os.path.join(INPUT_ROOT, "*wheelhouse*"),
        os.path.join(INPUT_ROOT, "*wheels*"),
        os.path.join(INPUT_ROOT, "*wheel*", "wheels"),
        os.path.join(INPUT_ROOT, "*", "wheels"),
        os.path.join(INPUT_ROOT, "*", "wheelhouse"),
    ]
    hits = []
    for pattern in patterns:
        hits.extend(glob.glob(pattern))
    for hit in sorted(set(hits)):
        if os.path.isdir(hit) and (glob.glob(os.path.join(hit, "vllm*.whl")) or glob.glob(os.path.join(hit, "*.whl"))):
            return hit
    return None


def vllm_present():
    try:
        mod = importlib.import_module("vllm")
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return None


def ensure_vllm():
    version = vllm_present()
    if version:
        log("vllm already importable, version=%s" % version)
        return True

    wheelhouse = find_wheelhouse()
    if wheelhouse:
        log("installing vllm from wheelhouse %s" % wheelhouse)
        rc = sh([
            sys.executable, "-m", "pip", "install", "--no-index",
            "--find-links", wheelhouse, "vllm",
        ])
        if rc == 0 and vllm_present():
            return True
        log("wheelhouse install failed (rc=%s); falling back to pip" % rc)

    if IS_RERUN:
        log("ERROR: competition rerun has no internet and no usable wheelhouse was attached.")
        log("Attach a pre-baked vLLM wheelhouse dataset, or bake vLLM into a custom image.")
        return False

    log("installing vllm from PyPI (internet is on for a non-rerun save)")
    rc = sh([sys.executable, "-m", "pip", "install", "--upgrade", "vllm"])
    return rc == 0 and bool(vllm_present())


VLLM_OK = ensure_vllm()
log("vllm available: %s (version=%s)" % (VLLM_OK, vllm_present()))
if not VLLM_OK:
    log("Proceeding anyway: the harness will emit a placeholder submission for all tasks.")
print("elapsed %.0fs, remaining %.2f h" % (elapsed_s(), remaining_s() / 3600.0))


In [ ]:
"""Locate the Qwen3.8-27B HF checkpoint among the attached Kaggle datasets."""
import glob
import os

MODEL_SLUG_CANDIDATES = [
    "qwen3-8-27b-fp8",
    "qwen3-8-27b-fp8-hf-snapshot",
    "qwen3-8-27b-bf16",
    "qwen3-8-27b",
]


def is_hf_checkpoint(path):
    if not os.path.isdir(path):
        return False
    if not os.path.exists(os.path.join(path, "config.json")):
        return False
    weights = glob.glob(os.path.join(path, "*.safetensors")) + glob.glob(os.path.join(path, "*.bin"))
    return bool(weights)


def find_checkpoint():
    tried = []
    for slug in MODEL_SLUG_CANDIDATES:
        base = os.path.join(INPUT_ROOT, slug)
        tried.append(base)
        if is_hf_checkpoint(base):
            return base
        if os.path.isdir(base):
            for sub in sorted(os.listdir(base)):
                cand = os.path.join(base, sub)
                if is_hf_checkpoint(cand):
                    return cand
    if os.path.isdir(INPUT_ROOT):
        for entry in sorted(os.listdir(INPUT_ROOT)):
            cand = os.path.join(INPUT_ROOT, entry)
            if "qwen3" in entry.lower() and "27b" in entry.lower():
                if is_hf_checkpoint(cand):
                    return cand
                if os.path.isdir(cand):
                    for sub in sorted(os.listdir(cand)):
                        deep = os.path.join(cand, sub)
                        if is_hf_checkpoint(deep):
                            return deep
    raise RuntimeError("no HF checkpoint found; looked at %s" % tried)


def checkpoint_size_gb(path):
    total = 0
    for root, _dirs, files in os.walk(path):
        for name in files:
            try:
                total += os.path.getsize(os.path.join(root, name))
            except OSError:
                pass
    return total / 1e9


try:
    MODEL_PATH = find_checkpoint()
    size_gb = checkpoint_size_gb(MODEL_PATH)
    log("checkpoint: %s (%.1f GB)" % (MODEL_PATH, size_gb))
    if size_gb < 20:
        log("WARNING: %.1f GB is small for a 27B FP8/BF16 checkpoint - verify it is complete." % size_gb)
    if "gguf" in MODEL_PATH.lower() or "ollama" in MODEL_PATH.lower():
        log("ERROR: %s looks like an Ollama/GGUF archive. vLLM needs an HF checkpoint." % MODEL_PATH)
        MODEL_PATH = None
except RuntimeError as exc:
    MODEL_PATH = None
    log("checkpoint discovery failed: %s" % exc)

print("elapsed %.0fs, remaining %.2f h" % (elapsed_s(), remaining_s() / 3600.0))


In [ ]:
"""Launch vLLM through a flag-fallback ladder and wait for health.

Every flag here is version-sensitive and none of it could be tested without a GPU, so
the ladder degrades rather than fails: TP=4 with full caching, then progressively
fewer optional flags, then TP=2 as a last resort. `--mamba-cache-mode align` (NOT
`all`) is required for prefix caching on the hybrid Gated DeltaNet architecture.
"""
import json
import os
import signal
import subprocess
import time
import urllib.request

LAUNCH_BUDGET = max(600.0, min(LAUNCH_RESERVE_S, remaining_s() - 3600.0 - SUBMISSION_BUFFER_S))
FIRST_RUNG_FRACTION = 0.45

RUNGS = [
    ("tp4-full", [
        "--tensor-parallel-size", "4", "--max-model-len", "32768",
        "--gpu-memory-utilization", "0.92", "--max-num-seqs", "64",
        "--reasoning-parser", "qwen3",
        "--enable-prefix-caching", "--mamba-cache-mode", "align",
    ], 1.0),
    ("tp4-noalign", [
        "--tensor-parallel-size", "4", "--max-model-len", "32768",
        "--gpu-memory-utilization", "0.92", "--max-num-seqs", "64",
        "--reasoning-parser", "qwen3", "--enable-prefix-caching",
    ], 0.6),
    ("tp4-nocache", [
        "--tensor-parallel-size", "4", "--max-model-len", "32768",
        "--gpu-memory-utilization", "0.92", "--max-num-seqs", "64",
        "--reasoning-parser", "qwen3",
    ], 0.6),
    ("tp4-bare", [
        "--tensor-parallel-size", "4", "--max-model-len", "32768",
        "--gpu-memory-utilization", "0.93", "--max-num-seqs", "64",
    ], 0.6),
    ("tp2-small", [
        "--tensor-parallel-size", "2", "--max-model-len", "24576",
        "--gpu-memory-utilization", "0.95", "--max-num-seqs", "12",
        "--reasoning-parser", "qwen3",
    ], 0.6),
    ("tp2-bare", [
        "--tensor-parallel-size", "2", "--max-model-len", "24576",
        "--gpu-memory-utilization", "0.95", "--max-num-seqs", "12",
    ], 0.6),
]


def endpoint_healthy(base_url, timeout=10):
    try:
        req = urllib.request.Request(base_url + "/models")
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status == 200
    except Exception:
        return False


def kill(proc):
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except Exception:
        try:
            proc.terminate()
        except Exception:
            pass
    try:
        proc.wait(timeout=30)
    except Exception:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:
            pass


def tail(path, n=25):
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as fh:
            return "".join(fh.readlines()[-n:])
    except Exception:
        return "<no log>"


def launch_rung(name, extra_args, port, log_path, timeout):
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_PATH,
        "--served-model-name", SERVED_MODEL_NAME,
        "--trust-remote-code", "--disable-log-requests",
        "--port", str(port),
    ] + extra_args
    log("rung %s: %s" % (name, " ".join(cmd[3:])))
    fh = open(log_path, "w", encoding="utf-8")
    proc = subprocess.Popen(
        cmd, stdout=fh, stderr=subprocess.STDOUT,
        preexec_fn=os.setsid if os.name != "nt" else None,
        env=dict(os.environ, VLLM_LOGGING_LEVEL="INFO"),
    )
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            log("rung %s: process exited rc=%s" % (name, proc.returncode))
            log(tail(log_path))
            fh.close()
            return None
        if endpoint_healthy("http://127.0.0.1:%d/v1" % port):
            log("rung %s: healthy after %.0fs" % (name, timeout - (deadline - time.monotonic())))
            return proc, fh
        time.sleep(5)
    log("rung %s: not healthy within %.0fs" % (name, timeout))
    log(tail(log_path))
    kill(proc)
    fh.close()
    return None


VLLM_PROC = None
VLLM_LOG_FH = None
SERVING = None

if MODEL_PATH and VLLM_OK:
    weights = [w for _n, _a, w in RUNGS]
    total_w = sum(weights)
    for i, (name, args, weight) in enumerate(RUNGS):
        share = FIRST_RUNG_FRACTION if i == 0 else (1.0 - FIRST_RUNG_FRACTION) * weight / (total_w - weights[0])
        timeout = max(180.0, LAUNCH_BUDGET * share)
        if remaining_s() - timeout < 3600.0 + SUBMISSION_BUFFER_S:
            log("not enough time left for rung %s (%.0fs); stopping the ladder" % (name, timeout))
            break
        result = launch_rung(name, args, VLLM_PORT, VLLM_LOG, timeout)
        if result is not None:
            VLLM_PROC, VLLM_LOG_FH = result
            SERVING = {"rung": name, "tp": args[args.index("--tensor-parallel-size") + 1], "base_url": BASE_URL}
            break

if SERVING:
    log("SERVING via %s at %s" % (SERVING["rung"], SERVING["base_url"]))
else:
    log("NO SERVER: every rung failed. The run will emit a schema-valid placeholder submission.")
print("elapsed %.0fs, remaining %.2f h" % (elapsed_s(), remaining_s() / 3600.0))


In [ ]:
"""Probe the live server: detect the reasoning_effort mechanism and calibrate token cost.

Two things cannot be known without a GPU and both are measured here instead of guessed:

1. *How* `reasoning_effort` reaches the Qwen3.8 chat template. The documented
   mechanism is `chat_template_kwargs`; some vLLM builds want it top-level. A wrong
   guess is an HTTP 500 on every request, so all three are tried once, in order.
2. *How long* one xhigh sample actually is. The budget assumes 10,000 completion
   tokens; the real mean over 2 probes replaces it, which directly rescales how many
   xhigh samples per task the governor will approve.
"""
import time

from harness.config import HarnessConfig, SamplingConfig, ServingConfig
from harness.llm import ChatRequest, HTTPChatClient
from harness.parse import merge_reasoning_and_content, parse_answers
from harness.prompts import build_messages
from harness.tasks import load_challenges, resolve_solutions_path

PROBE_EFFORTS = ("low", "medium", "xhigh")
MECHANISMS = ("chat_template_kwargs", "top_level", "none")

client = None
EFFORT_MECHANISM = None
CALIBRATED_XHIGH_TOKENS = XHIGH_TOKENS_PER_SAMPLE
probe_rows = []

if SERVING:
    challenges = os.path.join(DATA_DIR, "arc-agi_%s_challenges.json" % SPLIT)
    solutions = resolve_solutions_path(DATA_DIR, SPLIT)
    probe_tasks = load_challenges(challenges, solutions)[:1]

    for mechanism in MECHANISMS:
        candidate = HTTPChatClient(
            base_url=SERVING["base_url"], model=SERVED_MODEL_NAME,
            timeout=1800.0, max_retries=1, effort_mechanism=mechanism,
        )
        resp = candidate.complete(ChatRequest(
            messages=[{"role": "user", "content": "Reply with the single digit 7."}],
            max_tokens=128, temperature=0.0, reasoning_effort="low",
        ))
        if resp.ok:
            client = candidate
            EFFORT_MECHANISM = mechanism
            log("reasoning_effort mechanism = %s" % mechanism)
            break
        log("mechanism %s failed: %s" % (mechanism, (resp.error or "")[:200]))

    if client is None:
        log("no effort mechanism worked; the server is unusable")

if client is not None and probe_tasks:
    task = probe_tasks[0]
    messages = build_messages(task)
    for effort in PROBE_EFFORTS:
        started = time.monotonic()
        resp = client.complete(ChatRequest(
            messages=messages,
            max_tokens=XHIGH_MAX_TOKENS if effort == "xhigh" else PHASE_A_MAX_TOKENS,
            temperature=XHIGH_TEMPERATURE if effort == "xhigh" else 0.7,
            top_p=TOP_P, seed=1, reasoning_effort=effort,
            meta={"task_id": task.task_id, "probe": True},
        ))
        wall = time.monotonic() - started
        parsed = parse_answers(merge_reasoning_and_content(resp.reasoning, resp.content), task.n_test)
        tps = resp.usage.completion_tokens / wall if wall > 0 else 0.0
        probe_rows.append({
            "effort": effort, "ok": resp.ok, "error": resp.error, "wall_s": round(wall, 1),
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "reasoning_tokens": resp.usage.reasoning_tokens,
            "single_stream_tok_s": round(tps, 1),
            "finish_reason": resp.finish_reason,
            "parse_method": parsed.method, "n_grids": len(parsed.grids),
        })
        log("probe %-6s ok=%s wall=%6.1fs completion=%6d reasoning=%6d %.1f tok/s parse=%s(%d)" % (
            effort, resp.ok, wall, resp.usage.completion_tokens,
            resp.usage.reasoning_tokens, tps, parsed.method, len(parsed.grids)))
        if effort == "xhigh" and resp.ok and resp.usage.completion_tokens > 0:
            measured = resp.usage.completion_tokens
            CALIBRATED_XHIGH_TOKENS = int(min(max(measured, 4000), XHIGH_MAX_TOKENS))
            log("calibrated xhigh_tokens_per_sample: %d -> %d" % (XHIGH_TOKENS_PER_SAMPLE, CALIBRATED_XHIGH_TOKENS))
            if resp.finish_reason == "length":
                log("WARNING: xhigh hit max_tokens=%d; expect truncated answers" % XHIGH_MAX_TOKENS)

PROBE_JSON = probe_rows
log("probe complete; elapsed %.0fs, remaining %.2f h" % (elapsed_s(), remaining_s() / 3600.0))


In [ ]:
"""Run the H1 harness over the competition test split with a hard watchdog armed.

The governor's horizon is rebased on time *already spent*: `total_seconds` is set to
what remains of the 12 h and `startup_seconds` to 0, so `t_gen`/`t_submit`/`t_hard`
land on the true absolute deadlines rather than 12 h from now.
"""
import json

from harness.budget import BudgetConfig, BudgetGovernor, SystemClock
from harness.config import HarnessConfig, SamplingConfig, ServingConfig
from harness.llm import HTTPChatClient
from harness.runner import default_logger, load_tasks, run_harness

HARNESS_LOG = default_logger("h1")
tasks = load_tasks(HarnessConfig(data_dir=DATA_DIR, split=SPLIT, limit=LIMIT))
HARNESS_LOG("loaded %d tasks / %d test outputs from split=%s" % (
    len(tasks), sum(t.n_test for t in tasks), SPLIT))

clock = SystemClock()
budget = BudgetConfig(
    total_seconds=max(600.0, remaining_s()),
    submission_buffer_seconds=SUBMISSION_BUFFER_S,
    startup_seconds=0.0,
    finalize_reserve_seconds=FINALIZE_RESERVE_S,
    safety_reserve_fraction=SAFETY_RESERVE_FRACTION,
    planned_tok_per_s=PLANNED_TOK_S,
    pessimistic_tok_per_s=PESSIMISTIC_TOK_S,
    optimistic_tok_per_s=OPTIMISTIC_TOK_S,
    xhigh_tokens_per_sample=CALIBRATED_XHIGH_TOKENS,
    low_tokens_per_sample=LOW_TOKENS_PER_SAMPLE,
)
governor = BudgetGovernor(budget, clock=clock)
HARNESS_LOG("deadline: t_gen=+%.0fs t_submit=+%.0fs planned_window=%.0fs projected=%.0f tok/s" % (
    governor.t_gen - governor.t0, governor.t_submit - governor.t0,
    governor.plan_seconds_left(), governor.projected_tok_per_s()))

config = HarnessConfig(data_dir=DATA_DIR, split=SPLIT, output_dir=WORKING, limit=LIMIT)
config.serving = ServingConfig(
    model=SERVED_MODEL_NAME,
    base_url=SERVING["base_url"] if SERVING else BASE_URL,
    effort_mechanism=EFFORT_MECHANISM or "chat_template_kwargs",
)
config.sampling = SamplingConfig(
    phase_a_effort=PHASE_A_EFFORT,
    phase_a_samples=PHASE_A_SAMPLES if client is not None else 0,
    phase_a_max_tokens=PHASE_A_MAX_TOKENS,
    phase_b_effort="xhigh",
    wave_samples=WAVE_SAMPLES if client is not None else (),
    max_xhigh_samples=MAX_XHIGH_SAMPLES,
    xhigh_max_tokens=XHIGH_MAX_TOKENS,
    xhigh_temperature=XHIGH_TEMPERATURE,
    top_p=TOP_P,
    consensus_threshold=CONSENSUS_THRESHOLD,
    consensus_min_candidates=CONSENSUS_MIN_CANDIDATES,
    concurrency=CONCURRENCY,
    shuffle_train=SHUFFLE_TRAIN,
    request_timeout=min(2400.0, max(300.0, governor.plan_seconds_left())),
)
config.budget = budget

if client is None:
    HARNESS_LOG("no usable LLM client; writing a schema-valid placeholder submission for all tasks")

outcome = run_harness(
    config,
    client=client,
    clock=clock,
    logger=HARNESS_LOG,
    tasks=tasks,
    governor=governor,
    submission_path=os.path.join(WORKING, "submission.json"),
    use_watchdog=True,
    hard_exit=True,
)

REPORT = outcome.report
SCORE = outcome.score
HARNESS_LOG("run finished in %.0fs; submission=%s" % (REPORT.elapsed_s, REPORT.submission_path))
with open(os.path.join(WORKING, "run_report.json"), "w", encoding="utf-8") as fh:
    json.dump(REPORT.as_dict(), fh, indent=2, default=str)
with open(os.path.join(WORKING, "probe.json"), "w", encoding="utf-8") as fh:
    json.dump(PROBE_JSON, fh, indent=2)


In [ ]:
"""Final validation and report. Self-scores in non-rerun mode (evaluation split has truth)."""
import json
import os

from harness.scoring import score_submission
from harness.submission import SubmissionWriter

SUBMISSION_PATH = os.path.join(WORKING, "submission.json")
expected_ids = {t.task_id for t in tasks}
expected_outputs = sum(t.n_test for t in tasks)

if not os.path.exists(SUBMISSION_PATH):
    log("FATAL: %s does not exist" % SUBMISSION_PATH)
else:
    blob = json.load(open(SUBMISSION_PATH, encoding="utf-8"))
    writer = SubmissionWriter(SUBMISSION_PATH, tasks)
    writer.data = blob
    errors = writer.validate()
    n_outputs = sum(len(v) for v in blob.values())
    log("submission: %d task_ids, %d test-output entries (expected %d / %d)" % (
        len(blob), n_outputs, len(expected_ids), expected_outputs))
    log("missing task_ids: %d" % len(expected_ids - set(blob)))
    log("schema errors: %d %s" % (len(errors), errors[:5]))
    log("size: %.2f MB" % (os.path.getsize(SUBMISSION_PATH) / 1e6))

    report = REPORT.as_dict()
    print()
    print("=" * 78)
    print("H1 pure-xhigh run summary")
    print("=" * 78)
    print("serving rung          : %s" % (SERVING["rung"] if SERVING else "NONE"))
    print("effort mechanism      : %s" % EFFORT_MECHANISM)
    print("tasks / test outputs  : %d / %d" % (report["n_tasks"], report["n_test_outputs"]))
    print("wall clock            : %.0f s" % report["elapsed_s"])
    print("samples/task          : %.2f (xhigh %.2f)" % (
        report["samples_per_task"], report["xhigh_samples_per_task"]))
    print("completion tokens     : %s" % format(report["tokens"]["completion_tokens"], ","))
    print("reasoning tokens      : %s" % format(report["tokens"]["reasoning_tokens"], ","))
    print("requests / failures   : %s / %s" % (report["tokens"]["requests"], report["tokens"]["failures"]))
    print("measured tok/s        : %s" % report["measured_tok_per_s"])
    print("planned tok/s         : %.0f (pessimistic %.0f)" % (PLANNED_TOK_S, PESSIMISTIC_TOK_S))
    print("consensus-stopped     : %d" % report["consensus_stopped_tasks"])
    print("tasks with no parse   : %d" % report["tasks_with_no_parse"])
    print("parse methods         : %s" % report["parse_method_counts"])
    for wave in report["waves"]:
        print("  wave %-4s effort=%-6s tasks=%3d reqs=%4d tok=%10s %7.1fs consensus=%3d%s" % (
            wave["phase"], wave["effort"], wave["tasks"], wave["requests"],
            format(wave["completion_tokens"], ","), wave["seconds"], wave["consensus_after"],
            (" [%s]" % wave["trimmed_reason"]) if wave["trimmed_reason"] else ""))

    if SCORE is not None:
        print("-" * 78)
        print("SCORE                 : %.2f%%  (%d/%d outputs)" % (
            SCORE.score * 100, SCORE.total_correct, SCORE.total_outputs))
        print("  attempt_1 only      : %.2f%%" % (SCORE.attempt_1_score * 100))
        print("  2nd-attempt gain    : %.2f pts" % (SCORE.second_attempt_gain * 100))
    if report.get("coverage"):
        cov = report["coverage"]
        print("-" * 78)
        print("pass@1 (per sample)   : %.2f%%" % (cov["pass_at_1"] * 100))
        print("coverage (pass@N)     : %.2f%%   <- HARD CEILING" % (cov["coverage_pass_at_n"] * 100))
        print("uncovered outputs     : %d" % cov["uncovered_outputs"])
        print("selection efficiency  : %.1f%%" % (cov["selection_efficiency"] * 100))
        print("VERDICT               : %s" % cov["verdict"])
    print("=" * 78)

    log("remaining %.2f h (submission buffer is %.0f s)" % (remaining_s() / 3600.0, SUBMISSION_BUFFER_S))
